# Generation Quality Evaluation

This notebook supports the final-submission evaluation of non-authoritative generated analyst guidance.

It will be used to:

- inspect the completed development workflow results,
- confirm that a balanced generation-evaluation sample is feasible,
- select and freeze a 12-case development subset,
- prepare the human-review artifact,
- run the LLM-as-judge evaluation,
- compare human and judge scores,
- generate reproducible evaluation artifacts.

This notebook does not use held-out claims during evaluation design or calibration.

Deterministic policy rules and triage outcomes remain authoritative.

## 1. Environment Bootstrap

This section resolves the repository root, adds it to the Python path, and verifies the expected project structure before loading evaluation artifacts.

In [1]:
from pathlib import Path
import sys
import pandas as pd

def find_project_root(start_path: Path) -> Path:
    """Find the repository root by locating the src and data folders."""
    current = start_path.resolve()

    for candidate in [current, *current.parents]:
        if (
            (candidate / "src").is_dir()
            and (candidate / "data").is_dir()
            and (candidate / "notebooks").is_dir()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root containing src, data, and notebooks."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Python executable:", sys.executable)
print("Python version:", sys.version.split()[0])

Project root: /Users/sadiqsheik/Documents/MS/IIITB_Course Docs/Capstone Project/BYOC/DP_claims_triage
Python executable: /opt/anaconda3/envs/dpclaims/bin/python
Python version: 3.11.15


In [2]:
required_paths = {
    "source folder": PROJECT_ROOT / "src",
    "development workflow results": (
        PROJECT_ROOT
        / "data"
        / "evaluation"
        / "workflow"
        / "workflow_development_per_claim_results_v1.csv"
    ),
    "development workflow manifest": (
        PROJECT_ROOT
        / "data"
        / "evaluation"
        / "workflow"
        / "workflow_development_evaluation_manifest_v1.json"
    ),
    "generation evaluation design": (
        PROJECT_ROOT
        / "docs"
        / "generation_evaluation_design.md"
    ),
}

for label, path in required_paths.items():
    assert path.exists(), f"Missing {label}: {path}"
    print(f"PASS: {label} -> {path.relative_to(PROJECT_ROOT)}")

PASS: source folder -> src
PASS: development workflow results -> data/evaluation/workflow/workflow_development_per_claim_results_v1.csv
PASS: development workflow manifest -> data/evaluation/workflow/workflow_development_evaluation_manifest_v1.json
PASS: generation evaluation design -> docs/generation_evaluation_design.md


## 2. Load Development Workflow Results

This section loads only the completed development-set workflow evaluation artifact.

The held-out claim set and held-out labels are not accessed.

In [3]:
import json

import pandas as pd

WORKFLOW_RESULTS_PATH = required_paths["development workflow results"]
WORKFLOW_MANIFEST_PATH = required_paths["development workflow manifest"]

workflow_results = pd.read_csv(WORKFLOW_RESULTS_PATH)

with WORKFLOW_MANIFEST_PATH.open("r", encoding="utf-8") as file:
    workflow_manifest = json.load(file)

print("Workflow result shape:", workflow_results.shape)
print("Workflow manifest keys:", sorted(workflow_manifest.keys()))

Workflow result shape: (165, 24)
Workflow manifest keys: ['artifact_name', 'artifact_version', 'created_at_utc', 'evaluation_scope', 'headline_metrics', 'held_out_boundary', 'interpretation', 'label_source', 'mismatch_summary', 'output_files']


In [4]:
print("Available columns:")

for column in workflow_results.columns:
    print(f"- {column}")

Available columns:
- claim_id
- workflow_status
- gold_disposition
- predicted_disposition
- disposition_match
- gold_primary_rule_id
- gold_acceptable_primary_rule_ids
- predicted_triggering_rule_id
- primary_rule_exact_match
- primary_rule_acceptable_match
- deterministic_outcome
- final_matches_deterministic_outcome
- deterministic_rule
- final_matches_deterministic_rule
- authority_guardrail_status
- content_safety_status
- content_safety_fallback_used
- gold_follow_up_question_ids
- predicted_follow_up_question_ids
- follow_up_exact_match
- predicted_follow_up_required
- predicted_follow_up_selection_status
- decision_support_only
- workflow_trace


## 3. Identify Candidate Outcome and Evaluation Columns

This section identifies columns that may support balanced case selection and generation-quality evaluation.

In [5]:
candidate_outcome_columns = [
    column
    for column in workflow_results.columns
    if any(
        keyword in column.lower()
        for keyword in ["outcome", "disposition", "triage"]
    )
]

print("Candidate outcome columns:")

for column in candidate_outcome_columns:
    print(f"- {column}")

Candidate outcome columns:
- gold_disposition
- predicted_disposition
- disposition_match
- deterministic_outcome
- final_matches_deterministic_outcome


In [6]:
for column in candidate_outcome_columns:
    print(f"\nValue counts for: {column}")

    display(
        workflow_results[column]
        .fillna("<MISSING>")
        .value_counts(dropna=False)
        .rename_axis(column)
        .reset_index(name="count")
    )


Value counts for: gold_disposition


,gold_disposition,count
0,PROCEED,53
1,INFO_REQUIRED,45
2,MANUAL_REVIEW,41
3,NOT_ELIGIBLE,26



Value counts for: predicted_disposition


,predicted_disposition,count
0,PROCEED,64
1,INFO_REQUIRED,45
2,MANUAL_REVIEW,31
3,NOT_ELIGIBLE,25



Value counts for: disposition_match


,disposition_match,count
0,True,151
1,False,14



Value counts for: deterministic_outcome


,deterministic_outcome,count
0,PROCEED,64
1,INFO_REQUIRED,45
2,MANUAL_REVIEW,31
3,NOT_ELIGIBLE,25



Value counts for: final_matches_deterministic_outcome


,final_matches_deterministic_outcome,count
0,True,165


In [7]:
inspection_keywords = [
    "claim",
    "outcome",
    "disposition",
    "rule",
    "type",
    "incident",
    "evidence",
    "manual",
    "retriev",
    "guidance",
    "explanation",
    "follow",
    "reason",
]

inspection_columns = [
    column
    for column in workflow_results.columns
    if any(keyword in column.lower() for keyword in inspection_keywords)
]

print("Likely useful columns:")

for column in inspection_columns:
    print(f"- {column}")

Likely useful columns:
- claim_id
- gold_disposition
- predicted_disposition
- disposition_match
- gold_primary_rule_id
- gold_acceptable_primary_rule_ids
- predicted_triggering_rule_id
- primary_rule_exact_match
- primary_rule_acceptable_match
- deterministic_outcome
- final_matches_deterministic_outcome
- deterministic_rule
- final_matches_deterministic_rule
- gold_follow_up_question_ids
- predicted_follow_up_question_ids
- follow_up_exact_match
- predicted_follow_up_required
- predicted_follow_up_selection_status


In [8]:
display(workflow_results[inspection_columns].head(10))

,claim_id,gold_disposition,predicted_disposition,disposition_match,gold_primary_rule_id,gold_acceptable_primary_rule_ids,predicted_triggering_rule_id,primary_rule_exact_match,primary_rule_acceptable_match,deterministic_outcome,final_matches_deterministic_outcome,deterministic_rule,final_matches_deterministic_rule,gold_follow_up_question_ids,predicted_follow_up_question_ids,follow_up_exact_match,predicted_follow_up_required,predicted_follow_up_selection_status
0,CLM-000001,PROCEED,PROCEED,True,OUT-001,['OUT-001'],OUT-001,True,True,PROCEED,True,OUT-001,True,[],[],True,False,NOT_REQUIRED
1,CLM-000002,PROCEED,PROCEED,True,OUT-001,['OUT-001'],OUT-001,True,True,PROCEED,True,OUT-001,True,[],[],True,False,NOT_REQUIRED
2,CLM-000003,PROCEED,PROCEED,True,OUT-001,['OUT-001'],OUT-001,True,True,PROCEED,True,OUT-001,True,[],[],True,False,NOT_REQUIRED
3,CLM-000004,PROCEED,PROCEED,True,OUT-001,['OUT-001'],OUT-001,True,True,PROCEED,True,OUT-001,True,[],[],True,False,NOT_REQUIRED
4,CLM-000005,PROCEED,PROCEED,True,OUT-001,['OUT-001'],OUT-001,True,True,PROCEED,True,OUT-001,True,[],[],True,False,NOT_REQUIRED
5,CLM-000006,PROCEED,PROCEED,True,OUT-001,['OUT-001'],OUT-001,True,True,PROCEED,True,OUT-001,True,[],[],True,False,NOT_REQUIRED
6,CLM-000007,PROCEED,PROCEED,True,OUT-001,['OUT-001'],OUT-001,True,True,PROCEED,True,OUT-001,True,[],[],True,False,NOT_REQUIRED
7,CLM-000008,PROCEED,PROCEED,True,OUT-001,['OUT-001'],OUT-001,True,True,PROCEED,True,OUT-001,True,[],[],True,False,NOT_REQUIRED
8,CLM-000009,PROCEED,PROCEED,True,OUT-001,['OUT-001'],OUT-001,True,True,PROCEED,True,OUT-001,True,[],[],True,False,NOT_REQUIRED
9,CLM-000010,PROCEED,PROCEED,True,OUT-001,['OUT-001'],OUT-001,True,True,PROCEED,True,OUT-001,True,[],[],True,False,NOT_REQUIRED


In [9]:
missing_summary = (
    workflow_results[inspection_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .rename("missing_count")
    .to_frame()
)

missing_summary["missing_percentage"] = (
    missing_summary["missing_count"] / len(workflow_results) * 100
).round(1)

display(missing_summary)

,missing_count,missing_percentage
claim_id,0,0.0
gold_disposition,0,0.0
predicted_follow_up_required,0,0.0
follow_up_exact_match,0,0.0
predicted_follow_up_question_ids,0,0.0
gold_follow_up_question_ids,0,0.0
final_matches_deterministic_rule,0,0.0
deterministic_rule,0,0.0
final_matches_deterministic_outcome,0,0.0
deterministic_outcome,0,0.0


## 4. Sample Feasibility Check

The planned generation-quality sample contains 12 development cases:

- 3 `PROCEED`
- 3 `REQUEST_INFO`
- 3 `MANUAL_REVIEW`
- 3 `REJECT`

The next cells will validate whether each required outcome has enough development cases.

The final sampling column will be selected only after reviewing the available outcome fields.

In [10]:
TARGET_SAMPLE_COUNTS = {
    "PROCEED": 3,
    "REQUEST_INFO": 3,
    "MANUAL_REVIEW": 3,
    "REJECT": 3,
}

print("Target sample distribution:")

for outcome, required_count in TARGET_SAMPLE_COUNTS.items():
    print(f"- {outcome}: {required_count}")

Target sample distribution:
- PROCEED: 3
- REQUEST_INFO: 3
- MANUAL_REVIEW: 3
- REJECT: 3


## 5. Inspect Workflow Trace Structure

The per-claim workflow results contain a serialized `workflow_trace` field.

This section inspects its structure to determine whether the deterministic facts, retrieval results, and generated analyst-facing content can be reconstructed from the existing evaluation artifact or must be regenerated through the frozen workflow.

In [11]:
import ast
import json


def parse_serialized_value(value):
    """Parse a JSON or Python-literal string without executing arbitrary code."""
    if isinstance(value, (dict, list)):
        return value

    if pd.isna(value):
        return None

    text = str(value).strip()

    if not text:
        return None

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        try:
            return ast.literal_eval(text)
        except (ValueError, SyntaxError):
            return text


parsed_workflow_traces = workflow_results["workflow_trace"].apply(
    parse_serialized_value
)

print("Parsed trace value types:")
display(
    parsed_workflow_traces
    .apply(lambda value: type(value).__name__)
    .value_counts()
    .rename_axis("value_type")
    .reset_index(name="count")
)

Parsed trace value types:


,value_type,count
0,str,165


In [12]:
first_trace = parsed_workflow_traces.iloc[0]

print("First trace type:", type(first_trace).__name__)

if isinstance(first_trace, dict):
    print("\nTop-level keys:")
    for key in first_trace:
        print(f"- {key}")
elif isinstance(first_trace, list):
    print("\nNumber of trace entries:", len(first_trace))

    for index, entry in enumerate(first_trace[:10]):
        print(f"\nEntry {index}:")
        print("Type:", type(entry).__name__)

        if isinstance(entry, dict):
            print("Keys:", sorted(entry.keys()))
            display(entry)
        else:
            print(entry)
else:
    print(first_trace)

First trace type: str
deterministic_triage -> controlled_follow_up_selection -> explanation_proposal -> agent_content_safety_guardrail -> response_guardrail


## 6. Freeze the 12-Case Development Sample


This section selects a balanced 12-case development subset using the workflow-produced `deterministic_outcome` field.

The sample is frozen before any human scoring or LLM judging.

In [13]:
SAMPLING_OUTCOME_COLUMN = "deterministic_outcome"

TARGET_SAMPLE_COUNTS = {
    "PROCEED": 3,
    "INFO_REQUIRED": 3,
    "MANUAL_REVIEW": 3,
    "NOT_ELIGIBLE": 3,
}

available_counts = (
    workflow_results[SAMPLING_OUTCOME_COLUMN]
    .value_counts()
    .to_dict()
)

for outcome, required_count in TARGET_SAMPLE_COUNTS.items():
    available_count = int(available_counts.get(outcome, 0))

    assert available_count >= required_count, (
        f"Insufficient cases for {outcome}: "
        f"required {required_count}, available {available_count}"
    )

print("PASS: balanced 12-case sample is feasible.")
print("Sampling column:", SAMPLING_OUTCOME_COLUMN)

PASS: balanced 12-case sample is feasible.
Sampling column: deterministic_outcome


### Deterministic Selection Rule

The sample uses a fixed random seed and stratified selection by deterministic outcome.

Within each outcome, three claims are sampled without replacement. The resulting claim IDs are then sorted by outcome and claim ID for stable review order.

This method balances outcome coverage while avoiding manual selection based on generated-content quality.

In [14]:
SAMPLE_RANDOM_SEED = 42

sample_parts = []

for outcome, sample_count in TARGET_SAMPLE_COUNTS.items():
    outcome_rows = workflow_results.loc[
        workflow_results[SAMPLING_OUTCOME_COLUMN].eq(outcome)
    ].copy()

    sampled_rows = outcome_rows.sample(
        n=sample_count,
        random_state=SAMPLE_RANDOM_SEED,
        replace=False,
    )

    sample_parts.append(sampled_rows)

generation_sample = pd.concat(
    sample_parts,
    ignore_index=True,
)

outcome_order = {
    "PROCEED": 1,
    "INFO_REQUIRED": 2,
    "MANUAL_REVIEW": 3,
    "NOT_ELIGIBLE": 4,
}

generation_sample["outcome_order"] = (
    generation_sample[SAMPLING_OUTCOME_COLUMN]
    .map(outcome_order)
)

generation_sample = (
    generation_sample
    .sort_values(
        ["outcome_order", "claim_id"],
        kind="stable",
    )
    .drop(columns=["outcome_order"])
    .reset_index(drop=True)
)

generation_sample.insert(
    0,
    "evaluation_case_id",
    [
        f"GEN-{index:03d}"
        for index in range(1, len(generation_sample) + 1)
    ],
)

display(
    generation_sample[
        [
            "evaluation_case_id",
            "claim_id",
            "deterministic_outcome",
            "predicted_triggering_rule_id",
            "predicted_follow_up_required",
            "predicted_follow_up_question_ids",
        ]
    ]
)

,evaluation_case_id,claim_id,deterministic_outcome,predicted_triggering_rule_id,predicted_follow_up_required,predicted_follow_up_question_ids
0,GEN-001,CLM-000001,PROCEED,OUT-001,False,[]
1,GEN-002,CLM-000053,PROCEED,OUT-001,False,[]
2,GEN-003,CLM-000177,PROCEED,OUT-001,False,[]
3,GEN-004,CLM-000103,INFO_REQUIRED,DEV-001,True,['FUP-003']
4,GEN-005,CLM-000104,INFO_REQUIRED,DEV-001,True,['FUP-003']
5,GEN-006,CLM-000122,INFO_REQUIRED,EVD-001,True,['FUP-011']
6,GEN-007,CLM-000149,MANUAL_REVIEW,EVD-002,False,[]
7,GEN-008,CLM-000164,MANUAL_REVIEW,DATA-002,False,[]
8,GEN-009,CLM-000180,MANUAL_REVIEW,ANM-001,False,[]
9,GEN-010,CLM-000154,NOT_ELIGIBLE,COV-001,False,[]


In [15]:
sample_distribution = (
    generation_sample[SAMPLING_OUTCOME_COLUMN]
    .value_counts()
    .reindex(TARGET_SAMPLE_COUNTS.keys())
    .rename_axis("deterministic_outcome")
    .reset_index(name="sample_count")
)

display(sample_distribution)

assert len(generation_sample) == 12

for outcome, expected_count in TARGET_SAMPLE_COUNTS.items():
    actual_count = int(
        generation_sample[SAMPLING_OUTCOME_COLUMN]
        .eq(outcome)
        .sum()
    )

    assert actual_count == expected_count, (
        f"Unexpected sample count for {outcome}: "
        f"expected {expected_count}, got {actual_count}"
    )

assert generation_sample["claim_id"].is_unique

print("PASS: 12-case sample is balanced and contains unique claim IDs.")

,deterministic_outcome,sample_count
0,PROCEED,3
1,INFO_REQUIRED,3
2,MANUAL_REVIEW,3
3,NOT_ELIGIBLE,3


PASS: 12-case sample is balanced and contains unique claim IDs.


## 7. Run the Frozen Sample Through the Existing Workflow

This section reruns only the 12 frozen development claims through the existing guarded LangGraph workflow.

Configuration:

- Deterministic triage remains authoritative
- Controlled RAG is enabled
- The persisted FAISS index is used
- Cross-encoder reranking is enabled
- The existing OpenAI structured explanation builder is used
- Final output still passes through the content-safety and response-authority guardrails

The purpose is to capture the complete evaluation payload. No workflow logic is modified.

In [16]:
# Reuse the existing workflow, explanation builder, FAISS index, and reranker.

import os
from functools import partial

from openai import OpenAI

from src.agent.langgraph_orchestrator import (
    run_langgraph_guarded_claim_triage,
)
from src.agent.openai_explainer import (
    DEFAULT_MODEL as DEFAULT_EXPLANATION_MODEL,
    build_openai_explanation_proposal,
)
from src.data_loader import load_runtime_data
from src.rag.cross_encoder_reranker import (
    DEFAULT_CROSS_ENCODER_MODEL,
    SentenceTransformersCrossEncoderScorer,
)

assert os.getenv("OPENAI_API_KEY"), (
    "OPENAI_API_KEY is not available in the notebook environment."
)

RAG_ARTIFACT_DIR = (
    PROJECT_ROOT
    / "data"
    / "artifacts"
    / "rag"
    / "faiss_semantic_index_v1"
)

assert RAG_ARTIFACT_DIR.is_dir(), (
    f"Persisted FAISS artifact directory not found: {RAG_ARTIFACT_DIR}"
)

GENERATION_EVALUATION_CONFIG = {
    "enable_controlled_rag": True,
    "rag_top_k": 3,
    "rag_min_relevance_score": 0.20,
    "enable_reranking": True,
    "rag_rerank_top_n": 3,
    "explanation_model": DEFAULT_EXPLANATION_MODEL,
    "reranker_model": DEFAULT_CROSS_ENCODER_MODEL,
}

print("Generation evaluation configuration:")

for key, value in GENERATION_EVALUATION_CONFIG.items():
    print(f"- {key}: {value}")

Generation evaluation configuration:
- enable_controlled_rag: True
- rag_top_k: 3
- rag_min_relevance_score: 0.2
- enable_reranking: True
- rag_rerank_top_n: 3
- explanation_model: gpt-5.4-mini
- reranker_model: cross-encoder/ms-marco-MiniLM-L-6-v2


In [17]:
# Initialise clients once so all 12 cases use the same configuration.

openai_client = OpenAI()

explanation_proposal_builder = partial(
    build_openai_explanation_proposal,
    model=GENERATION_EVALUATION_CONFIG["explanation_model"],
    client=openai_client,
)

reranker_scorer = SentenceTransformersCrossEncoderScorer(
    model_name=GENERATION_EVALUATION_CONFIG["reranker_model"],
)

runtime_data = load_runtime_data()

print("PASS: runtime data loaded.")
print("PASS: OpenAI client initialised.")
print("PASS: cross-encoder reranker initialised.")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

PASS: runtime data loaded.
PASS: OpenAI client initialised.
PASS: cross-encoder reranker initialised.


In [18]:
# Execute the frozen sample without changing production workflow logic.

generation_workflow_outputs = []
generation_execution_errors = []

for sample_row in generation_sample.itertuples(index=False):
    evaluation_case_id = sample_row.evaluation_case_id
    claim_id = sample_row.claim_id

    try:
        workflow_output = run_langgraph_guarded_claim_triage(
            data=runtime_data,
            claim_id=claim_id,
            proposal_builder=explanation_proposal_builder,
            enable_controlled_rag=True,
            rag_artifact_dir=RAG_ARTIFACT_DIR,
            rag_retrieval_client=openai_client,
            rag_top_k=GENERATION_EVALUATION_CONFIG["rag_top_k"],
            rag_min_relevance_score=(
                GENERATION_EVALUATION_CONFIG[
                    "rag_min_relevance_score"
                ]
            ),
            enable_reranking=True,
            rag_reranker_scorer=reranker_scorer,
            rag_rerank_top_n=(
                GENERATION_EVALUATION_CONFIG[
                    "rag_rerank_top_n"
                ]
            ),
            rag_reranker_model_name=(
                GENERATION_EVALUATION_CONFIG[
                    "reranker_model"
                ]
            ),
        )

        generation_workflow_outputs.append(
            {
                "evaluation_case_id": evaluation_case_id,
                "claim_id": claim_id,
                "workflow_output": workflow_output,
            }
        )

        print(
            f"PASS: {evaluation_case_id} / {claim_id} "
            f"-> {workflow_output['workflow_status']}"
        )

    except Exception as exc:
        generation_execution_errors.append(
            {
                "evaluation_case_id": evaluation_case_id,
                "claim_id": claim_id,
                "error_type": type(exc).__name__,
                "error_message": str(exc),
            }
        )

        print(
            f"FAIL: {evaluation_case_id} / {claim_id} "
            f"-> {type(exc).__name__}: {exc}"
        )

print("\nExecution summary:")
print("- Successful:", len(generation_workflow_outputs))
print("- Failed:", len(generation_execution_errors))

PASS: GEN-001 / CLM-000001 -> COMPLETED
PASS: GEN-002 / CLM-000053 -> COMPLETED
PASS: GEN-003 / CLM-000177 -> COMPLETED
PASS: GEN-004 / CLM-000103 -> COMPLETED
PASS: GEN-005 / CLM-000104 -> COMPLETED
PASS: GEN-006 / CLM-000122 -> COMPLETED
PASS: GEN-007 / CLM-000149 -> COMPLETED
PASS: GEN-008 / CLM-000164 -> COMPLETED
PASS: GEN-009 / CLM-000180 -> COMPLETED
PASS: GEN-010 / CLM-000154 -> COMPLETED
PASS: GEN-011 / CLM-000191 -> COMPLETED
PASS: GEN-012 / CLM-000206 -> COMPLETED

Execution summary:
- Successful: 12
- Failed: 0


In [19]:
# Do not continue to human review with incomplete workflow outputs.

if generation_execution_errors:
    display(pd.DataFrame(generation_execution_errors))

assert not generation_execution_errors, (
    "One or more frozen evaluation cases failed. "
    "Resolve the execution error before continuing."
)

assert len(generation_workflow_outputs) == 12

returned_claim_ids = {
    item["claim_id"]
    for item in generation_workflow_outputs
}

expected_claim_ids = set(generation_sample["claim_id"])

assert returned_claim_ids == expected_claim_ids

print("PASS: all 12 frozen claims completed successfully.")

PASS: all 12 frozen claims completed successfully.


In [20]:
# Confirm that generated content did not alter deterministic authority.

workflow_validation_rows = []

for item in generation_workflow_outputs:
    output = item["workflow_output"]

    deterministic_decision = output[
        "tool_result"
    ]["deterministic_decision"]

    final_response = output["final_response"]
    rag_tool_result = output["rag_tool_result"]
    analyst_guidance = output["analyst_guidance"]

    workflow_validation_rows.append(
        {
            "evaluation_case_id": item["evaluation_case_id"],
            "claim_id": item["claim_id"],
            "workflow_status": output["workflow_status"],
            "workflow_version": output["workflow_version"],
            "proposal_source": output["proposal_source"],
            "deterministic_outcome": deterministic_decision[
                "triage_outcome"
            ],
            "final_outcome": final_response["triage_outcome"],
            "deterministic_rule": deterministic_decision[
                "triggering_rule_id"
            ],
            "final_rule": final_response["triggering_rule_id"],
            "authority_guardrail_status": final_response[
                "authority_guardrail"
            ]["status"],
            "content_safety_status": output[
                "content_safety"
            ]["content_safety_status"],
            "content_safety_fallback_used": output[
                "content_safety"
            ]["fallback_used"],
            "retrieval_status": rag_tool_result.get(
                "retrieval_status"
            ),
            "retrieved_guidance_count": analyst_guidance.get(
                "retrieved_guidance_count"
            ),
            "reranking_status": rag_tool_result.get(
                "reranking",
                {},
            ).get("reranking_status"),
        }
    )

generation_workflow_validation = pd.DataFrame(
    workflow_validation_rows
)

display(generation_workflow_validation)

,evaluation_case_id,claim_id,workflow_status,workflow_version,proposal_source,deterministic_outcome,final_outcome,deterministic_rule,final_rule,authority_guardrail_status,content_safety_status,content_safety_fallback_used,retrieval_status,retrieved_guidance_count,reranking_status
0,GEN-001,CLM-000001,COMPLETED,langgraph_v6,CUSTOM,PROCEED,PROCEED,OUT-001,OUT-001,ALIGNED,SAFE,False,RESULTS_FOUND,3,RERANKED
1,GEN-002,CLM-000053,COMPLETED,langgraph_v6,CUSTOM,PROCEED,PROCEED,OUT-001,OUT-001,ALIGNED,SAFE,False,RESULTS_FOUND,3,RERANKED
2,GEN-003,CLM-000177,COMPLETED,langgraph_v6,CUSTOM,PROCEED,PROCEED,OUT-001,OUT-001,ALIGNED,SAFE,False,RESULTS_FOUND,3,RERANKED
3,GEN-004,CLM-000103,COMPLETED,langgraph_v6,CUSTOM,INFO_REQUIRED,INFO_REQUIRED,DEV-001,DEV-001,ALIGNED,SAFE,False,RESULTS_FOUND,3,RERANKED
4,GEN-005,CLM-000104,COMPLETED,langgraph_v6,CUSTOM,INFO_REQUIRED,INFO_REQUIRED,DEV-001,DEV-001,ALIGNED,SAFE,False,RESULTS_FOUND,3,RERANKED
5,GEN-006,CLM-000122,COMPLETED,langgraph_v6,CUSTOM,INFO_REQUIRED,INFO_REQUIRED,EVD-001,EVD-001,ALIGNED,SAFE,False,RESULTS_FOUND,3,RERANKED
6,GEN-007,CLM-000149,COMPLETED,langgraph_v6,CUSTOM,MANUAL_REVIEW,MANUAL_REVIEW,EVD-002,EVD-002,ALIGNED,SAFE,False,RESULTS_FOUND,3,RERANKED
7,GEN-008,CLM-000164,COMPLETED,langgraph_v6,CUSTOM,MANUAL_REVIEW,MANUAL_REVIEW,DATA-002,DATA-002,ALIGNED,SAFE,False,RESULTS_FOUND,3,RERANKED
8,GEN-009,CLM-000180,COMPLETED,langgraph_v6,CUSTOM,MANUAL_REVIEW,MANUAL_REVIEW,ANM-001,ANM-001,ALIGNED,SAFE,False,RESULTS_FOUND,3,RERANKED
9,GEN-010,CLM-000154,COMPLETED,langgraph_v6,CUSTOM,NOT_ELIGIBLE,NOT_ELIGIBLE,COV-001,COV-001,ALIGNED,SAFE,False,RESULTS_FOUND,3,RERANKED


In [21]:
assert generation_workflow_validation[
    "workflow_status"
].eq("COMPLETED").all()

assert generation_workflow_validation[
    "proposal_source"
].eq("CUSTOM").all()

assert (
    generation_workflow_validation["deterministic_outcome"]
    == generation_workflow_validation["final_outcome"]
).all()

assert (
    generation_workflow_validation["deterministic_rule"]
    == generation_workflow_validation["final_rule"]
).all()

assert generation_workflow_validation[
    "authority_guardrail_status"
].eq("ALIGNED").all()

assert generation_workflow_validation[
    "content_safety_status"
].isin(["SAFE", "FALLBACK_APPLIED"]).all()

assert generation_workflow_validation[
    "reranking_status"
].isin(["RERANKED", "NO_CANDIDATES"]).all()

print("PASS: deterministic outcome preserved for all cases.")
print("PASS: deterministic rule preserved for all cases.")
print("PASS: authority guardrail aligned for all cases.")
print("PASS: controlled RAG and reranking executed.")

PASS: deterministic outcome preserved for all cases.
PASS: deterministic rule preserved for all cases.
PASS: authority guardrail aligned for all cases.
PASS: controlled RAG and reranking executed.


## 8. Build and Save the Generation Evaluation Case Artifact

This section converts the 12 successful nested workflow outputs into a stable evaluation dataset.

The artifact preserves:

- authoritative deterministic decision fields,
- projected RAG facts,
- controlled query and retrieval provenance,
- retrieved KB passages,
- analyst-only guidance,
- generated explanation fields,
- guarded final response,
- content-safety and authority-control results.

Nested values are stored as JSON strings so the CSV remains portable and reviewable.

In [22]:
from collections.abc import Mapping
from typing import Any


def as_plain_dict(value: object) -> dict[str, Any]:
    """Return a plain dictionary when the value is mapping-like."""
    if isinstance(value, Mapping):
        return dict(value)

    return {}


def as_plain_list(value: object) -> list[Any]:
    """Return a plain list when the value is list-like."""
    if isinstance(value, list):
        return value

    return []


def to_json_text(value: object) -> str:
    """Serialise nested values to stable JSON text."""
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        default=str,
    )

In [23]:
generation_case_rows = []

for item in generation_workflow_outputs:
    output = as_plain_dict(item["workflow_output"])

    # Authoritative deterministic decision.
    tool_result = as_plain_dict(output.get("tool_result"))

    deterministic_decision = as_plain_dict(
        tool_result.get("deterministic_decision")
    )

    # Controlled follow-up output.
    follow_up_tool_result = as_plain_dict(
        output.get("follow_up_tool_result")
    )

    follow_up_selection = as_plain_dict(
        follow_up_tool_result.get("follow_up_selection")
    )

    # Controlled RAG output.
    rag_tool_result = as_plain_dict(
        output.get("rag_tool_result")
    )

    projected_facts = as_plain_dict(
        rag_tool_result.get("projected_facts")
    )

    controlled_query = as_plain_dict(
        rag_tool_result.get("controlled_query")
    )

    retrieved_guidance = as_plain_list(
        rag_tool_result.get("retrieved_guidance")
    )

    reranking = as_plain_dict(
        rag_tool_result.get("reranking")
    )

    # Analyst-only formatted RAG guidance.
    analyst_guidance = as_plain_dict(
        output.get("analyst_guidance")
    )

    source_references = as_plain_list(
        analyst_guidance.get("source_references")
    )

    guidance_items = as_plain_list(
        analyst_guidance.get("guidance_items")
    )

    # Generated explanation and protected final response.
    agent_proposal = as_plain_dict(
        output.get("agent_proposal")
    )

    content_safety = as_plain_dict(
        output.get("content_safety")
    )

    final_response = as_plain_dict(
        output.get("final_response")
    )

    authority_guardrail = as_plain_dict(
        final_response.get("authority_guardrail")
    )

    # Build retrieval provenance lists.
    retrieved_chunk_ids = []

    retrieved_document_ids = []

    retrieved_context_blocks = []

    for record in retrieved_guidance:
        if not isinstance(record, Mapping):
            continue

        document_id = str(
            record.get("document_id", "")
        ).strip()

        chunk_id = str(
            record.get("chunk_id", "")
        ).strip()

        section_title = str(
            record.get("section_title", "")
        ).strip()

        chunk_text = str(
            record.get("chunk_text", "")
        ).strip()

        if chunk_id:
            retrieved_chunk_ids.append(chunk_id)

        if document_id:
            retrieved_document_ids.append(document_id)

        source_label_parts = [
            value
            for value in [
                document_id,
                chunk_id,
                section_title,
            ]
            if value
        ]

        source_label = " / ".join(
            source_label_parts
        )

        if chunk_text:
            if source_label:
                retrieved_context_blocks.append(
                    f"[{source_label}] {chunk_text}"
                )
            else:
                retrieved_context_blocks.append(
                    chunk_text
                )

    # Build reviewer-friendly analyst guidance text.
    analyst_guidance_text_parts = []

    for guidance_item in guidance_items:
        if not isinstance(guidance_item, Mapping):
            continue

        guidance_preview = str(
            guidance_item.get(
                "guidance_preview",
                "",
            )
        ).strip()

        if guidance_preview:
            analyst_guidance_text_parts.append(
                guidance_preview
            )

    analyst_guidance_text = "\n\n".join(
        analyst_guidance_text_parts
    )

    # Build consolidated generated explanation text.
    generated_explanation_parts = [
        str(
            agent_proposal.get(
                "case_summary",
                "",
            )
        ).strip(),
        str(
            agent_proposal.get(
                "reviewer_note",
                "",
            )
        ).strip(),
        str(
            agent_proposal.get(
                "next_step_message",
                "",
            )
        ).strip(),
    ]

    generated_explanation = "\n\n".join(
        value
        for value in generated_explanation_parts
        if value
    )

    # Build consolidated guarded final explanation text.
    final_explanation_parts = [
        str(
            final_response.get(
                "case_summary",
                "",
            )
        ).strip(),
        str(
            final_response.get(
                "reviewer_note",
                "",
            )
        ).strip(),
        str(
            final_response.get(
                "next_step_message",
                "",
            )
        ).strip(),
    ]

    final_explanation = "\n\n".join(
        value
        for value in final_explanation_parts
        if value
    )

    generation_case_rows.append(
        {
            "evaluation_case_id": item[
                "evaluation_case_id"
            ],
            "claim_id": item[
                "claim_id"
            ],

            # Workflow metadata
            "workflow_name": output.get(
                "workflow_name"
            ),
            "workflow_version": output.get(
                "workflow_version"
            ),
            "workflow_status": output.get(
                "workflow_status"
            ),
            "proposal_source": output.get(
                "proposal_source"
            ),

            # Authoritative deterministic decision
            "deterministic_outcome": (
                deterministic_decision.get(
                    "triage_outcome"
                )
            ),
            "triggering_rule_id": (
                deterministic_decision.get(
                    "triggering_rule_id"
                )
            ),
            "precedence_stage": (
                deterministic_decision.get(
                    "precedence_stage"
                )
            ),
            "deterministic_reason": (
                deterministic_decision.get(
                    "decision_reason"
                )
            ),
            "decision_support_only": (
                deterministic_decision.get(
                    "decision_support_only"
                )
            ),
            "system_limitations": to_json_text(
                deterministic_decision.get(
                    "system_limitations",
                    [],
                )
            ),
            "rule_trace": to_json_text(
                deterministic_decision.get(
                    "rule_trace",
                    [],
                )
            ),

            # Projected controlled RAG facts
            "projected_triage_outcome": (
                projected_facts.get(
                    "triage_outcome"
                )
            ),
            "projected_triggering_rule_id": (
                projected_facts.get(
                    "triggering_rule_id"
                )
            ),
            "projected_precedence_stage": (
                projected_facts.get(
                    "precedence_stage"
                )
            ),
            "claim_category": (
                projected_facts.get(
                    "claim_category"
                )
            ),
            "requested_service_type": (
                projected_facts.get(
                    "requested_service_type"
                )
            ),
            "plan_configuration_status": (
                projected_facts.get(
                    "plan_configuration_status"
                )
            ),
            "product_scope_status": (
                projected_facts.get(
                    "product_scope_status"
                )
            ),
            "coverage_lookup_status": (
                projected_facts.get(
                    "coverage_lookup_status"
                )
            ),
            "covered_flag": (
                projected_facts.get(
                    "covered_flag"
                )
            ),
            "evidence_profile_id": (
                projected_facts.get(
                    "evidence_profile_id"
                )
            ),
            "evidence_assessment_status": (
                projected_facts.get(
                    "evidence_assessment_status"
                )
            ),
            "missing_required_evidence_codes": (
                to_json_text(
                    projected_facts.get(
                        "missing_required_evidence_codes",
                        [],
                    )
                )
            ),
            "unreadable_required_evidence_codes": (
                to_json_text(
                    projected_facts.get(
                        "unreadable_required_evidence_codes",
                        [],
                    )
                )
            ),
            "device_match_status": (
                projected_facts.get(
                    "device_match_status"
                )
            ),
            "risk_indicator_ids": to_json_text(
                projected_facts.get(
                    "risk_indicator_ids",
                    [],
                )
            ),
            "manual_review_reason_codes": (
                to_json_text(
                    projected_facts.get(
                        "manual_review_reason_codes",
                        [],
                    )
                )
            ),

            # Controlled follow-up
            "follow_up_required": (
                follow_up_selection.get(
                    "follow_up_required"
                )
            ),
            "follow_up_selection_status": (
                follow_up_selection.get(
                    "selection_status"
                )
            ),
            "follow_up_question_ids": (
                to_json_text(
                    follow_up_selection.get(
                        "question_ids",
                        [],
                    )
                )
            ),

            # Controlled retrieval
            "controlled_query_text": (
                controlled_query.get(
                    "query_text"
                )
            ),
            "controlled_query_fingerprint": (
                controlled_query.get(
                    "query_fingerprint"
                )
            ),
            "retrieval_status": (
                rag_tool_result.get(
                    "retrieval_status"
                )
            ),
            "retrieved_guidance_count": (
                rag_tool_result.get(
                    "retrieved_guidance_count"
                )
            ),
            "retrieved_chunk_ids": (
                to_json_text(
                    retrieved_chunk_ids
                )
            ),
            "retrieved_document_ids": (
                to_json_text(
                    retrieved_document_ids
                )
            ),
            "retrieved_context": "\n\n".join(
                retrieved_context_blocks
            ),
            "retrieved_guidance_json": (
                to_json_text(
                    retrieved_guidance
                )
            ),
            "reranking_status": (
                reranking.get(
                    "reranking_status"
                )
            ),
            "reranker_model_name": (
                reranking.get(
                    "reranker_model_name"
                )
            ),

            # Analyst-only guidance
            "analyst_guidance_summary": (
                analyst_guidance.get(
                    "guidance_summary"
                )
            ),
            "analyst_guidance_usage_boundary": (
                analyst_guidance.get(
                    "usage_boundary"
                )
            ),
            "analyst_guidance_text": (
                analyst_guidance_text
            ),
            "source_references_json": (
                to_json_text(
                    source_references
                )
            ),
            "analyst_guidance_items_json": (
                to_json_text(
                    guidance_items
                )
            ),

            # Generated explanation
            "generated_case_summary": (
                agent_proposal.get(
                    "case_summary"
                )
            ),
            "generated_reviewer_note": (
                agent_proposal.get(
                    "reviewer_note"
                )
            ),
            "generated_next_step_message": (
                agent_proposal.get(
                    "next_step_message"
                )
            ),
            "generated_explanation": (
                generated_explanation
            ),

            # Safety and guarded final response
            "content_safety_status": (
                content_safety.get(
                    "content_safety_status"
                )
            ),
            "content_safety_fallback_used": (
                content_safety.get(
                    "fallback_used"
                )
            ),
            "content_safety_violations": (
                to_json_text(
                    content_safety.get(
                        "content_safety_violations",
                        [],
                    )
                )
            ),
            "final_outcome": (
                final_response.get(
                    "triage_outcome"
                )
            ),
            "final_triggering_rule_id": (
                final_response.get(
                    "triggering_rule_id"
                )
            ),
            "final_case_summary": (
                final_response.get(
                    "case_summary"
                )
            ),
            "final_reviewer_note": (
                final_response.get(
                    "reviewer_note"
                )
            ),
            "final_next_step_message": (
                final_response.get(
                    "next_step_message"
                )
            ),
            "final_explanation": (
                final_explanation
            ),
            "authority_guardrail_status": (
                authority_guardrail.get(
                    "status"
                )
            ),
            "authority_guardrail_violations": (
                to_json_text(
                    authority_guardrail.get(
                        "violations",
                        [],
                    )
                )
            ),
            "workflow_trace": to_json_text(
                output.get(
                    "workflow_trace",
                    [],
                )
            ),
        }
    )

generation_evaluation_cases = pd.DataFrame(
    generation_case_rows
)

print(
    "Generation evaluation artifact shape:",
    generation_evaluation_cases.shape,
)

Generation evaluation artifact shape: (12, 63)


In [24]:
# Correct the guarded final-response field extraction.
# Generated explanation fields are nested under final_response["agent_content"].

corrected_final_response_rows = []

for item in generation_workflow_outputs:
    output = as_plain_dict(item["workflow_output"])

    final_response = as_plain_dict(
        output.get("final_response")
    )

    final_agent_content = as_plain_dict(
        final_response.get("agent_content")
    )

    authority_guardrail = as_plain_dict(
        final_response.get("authority_guardrail")
    )

    final_case_summary = str(
        final_agent_content.get(
            "case_summary",
            "",
        )
    ).strip()

    final_reviewer_note = str(
        final_agent_content.get(
            "reviewer_note",
            "",
        )
    ).strip()

    final_next_step_message = str(
        final_agent_content.get(
            "next_step_message",
            "",
        )
    ).strip()

    final_explanation = "\n\n".join(
        value
        for value in [
            final_case_summary,
            final_reviewer_note,
            final_next_step_message,
        ]
        if value
    )

    corrected_final_response_rows.append(
        {
            "evaluation_case_id": item[
                "evaluation_case_id"
            ],
            "final_outcome": final_response.get(
                "triage_outcome"
            ),
            "final_triggering_rule_id": (
                final_response.get(
                    "triggering_rule_id"
                )
            ),
            "final_case_summary": (
                final_case_summary
            ),
            "final_reviewer_note": (
                final_reviewer_note
            ),
            "final_next_step_message": (
                final_next_step_message
            ),
            "final_explanation": (
                final_explanation
            ),
            "authority_guardrail_status": (
                authority_guardrail.get(
                    "status"
                )
            ),
            "authority_guardrail_conflicting_fields": (
                to_json_text(
                    authority_guardrail.get(
                        "conflicting_authority_fields",
                        [],
                    )
                )
            ),
            "authority_guardrail_ignored_fields": (
                to_json_text(
                    authority_guardrail.get(
                        "ignored_agent_fields",
                        [],
                    )
                )
            ),
            "authority_guardrail_notice": (
                authority_guardrail.get(
                    "authority_notice"
                )
            ),
        }
    )

corrected_final_response = pd.DataFrame(
    corrected_final_response_rows
).set_index("evaluation_case_id")

generation_evaluation_cases = (
    generation_evaluation_cases
    .set_index("evaluation_case_id")
)

for column in corrected_final_response.columns:
    generation_evaluation_cases[column] = (
        corrected_final_response[column]
    )

generation_evaluation_cases = (
    generation_evaluation_cases
    .reset_index()
)

print(
    "Corrected guarded final-response fields:",
    len(generation_evaluation_cases),
)

display(
    generation_evaluation_cases[
        [
            "evaluation_case_id",
            "claim_id",
            "final_outcome",
            "final_triggering_rule_id",
            "final_case_summary",
            "final_reviewer_note",
            "final_next_step_message",
            "authority_guardrail_status",
        ]
    ]
)

Corrected guarded final-response fields: 12


,evaluation_case_id,claim_id,final_outcome,final_triggering_rule_id,final_case_summary,final_reviewer_note,final_next_step_message,authority_guardrail_status
0,GEN-001,CLM-000001,PROCEED,OUT-001,Claim CLM-000001 routed to PROCEED at preceden...,This is a system triage recommendation based o...,Proceed with the standard processing workflow ...,ALIGNED
1,GEN-002,CLM-000053,PROCEED,OUT-001,Claim CLM-000053 routed to PROCEED at preceden...,"This is a system triage recommendation, decisi...",Proceed with standard processing workflow for ...,ALIGNED
2,GEN-003,CLM-000177,PROCEED,OUT-001,Claim CLM-000177 routed to PROCEED at preceden...,This is a system triage recommendation only. A...,Proceed with standard processing in line with ...,ALIGNED
3,GEN-004,CLM-000103,INFO_REQUIRED,DEV-001,Claim CLM-000103 routed to INFO_REQUIRED at pr...,This is a system triage recommendation only. A...,"Obtain and record a usable device identifier, ...",ALIGNED
4,GEN-005,CLM-000104,INFO_REQUIRED,DEV-001,Claim CLM-000104 routed to INFO_REQUIRED at pr...,This is a system triage recommendation only. A...,Obtain a usable device identifier (for example...,ALIGNED
5,GEN-006,CLM-000122,INFO_REQUIRED,EVD-001,Claim CLM-000122 routed to INFO_REQUIRED at pr...,This is a system triage recommendation only. A...,Request only the specific missing evidence ide...,ALIGNED
6,GEN-007,CLM-000149,MANUAL_REVIEW,EVD-002,Claim CLM-000149 routed to MANUAL_REVIEW at pr...,This is a system triage recommendation only. A...,Send the claim to the manual review queue for ...,ALIGNED
7,GEN-008,CLM-000164,MANUAL_REVIEW,DATA-002,Claim CLM-000164 routed to MANUAL_REVIEW at pr...,This is a system triage recommendation only. A...,Proceed with manual review of the claim and re...,ALIGNED
8,GEN-009,CLM-000180,MANUAL_REVIEW,ANM-001,Claim CLM-000180 routed to MANUAL_REVIEW. The ...,This is a system triage recommendation only. A...,Proceed with analyst review under the manual r...,ALIGNED
9,GEN-010,CLM-000154,NOT_ELIGIBLE,COV-001,Claim CLM-000154 routed to NOT_ELIGIBLE at pre...,This is a system triage recommendation only. A...,Proceed with the standard analyst review workf...,ALIGNED


In [25]:
core_review_columns = [
    "evaluation_case_id",
    "claim_id",
    "deterministic_outcome",
    "triggering_rule_id",
    "claim_category",
    "requested_service_type",
    "coverage_lookup_status",
    "evidence_assessment_status",
    "device_match_status",
    "manual_review_reason_codes",
    "retrieved_guidance_count",
    "reranking_status",
    "content_safety_status",
    "authority_guardrail_status",
]

display(
    generation_evaluation_cases[
        core_review_columns
    ]
)

,evaluation_case_id,claim_id,deterministic_outcome,triggering_rule_id,claim_category,requested_service_type,coverage_lookup_status,evidence_assessment_status,device_match_status,manual_review_reason_codes,retrieved_guidance_count,reranking_status,content_safety_status,authority_guardrail_status
0,GEN-001,CLM-000001,PROCEED,OUT-001,SCREEN_DAMAGE,REPAIR,UNIQUE_COVERAGE_RECORD,SUFFICIENT,DEVICE_MATCH,[],3,RERANKED,SAFE,ALIGNED
1,GEN-002,CLM-000053,PROCEED,OUT-001,SCREEN_DAMAGE,REPAIR,UNIQUE_COVERAGE_RECORD,SUFFICIENT,DEVICE_MATCH,[],3,RERANKED,SAFE,ALIGNED
2,GEN-003,CLM-000177,PROCEED,OUT-001,SCREEN_DAMAGE,UNSPECIFIED,UNIQUE_COVERAGE_RECORD,SUFFICIENT,DEVICE_MATCH,[],3,RERANKED,SAFE,ALIGNED
3,GEN-004,CLM-000103,INFO_REQUIRED,DEV-001,LIQUID_DAMAGE,REPAIR,UNIQUE_COVERAGE_RECORD,SUFFICIENT,DEVICE_IDENTIFIER_MISSING,[],3,RERANKED,SAFE,ALIGNED
4,GEN-005,CLM-000104,INFO_REQUIRED,DEV-001,MECHANICAL_BREAKDOWN,REPAIR,UNIQUE_COVERAGE_RECORD,SUFFICIENT,DEVICE_IDENTIFIER_MISSING,[],3,RERANKED,SAFE,ALIGNED
5,GEN-006,CLM-000122,INFO_REQUIRED,EVD-001,MECHANICAL_BREAKDOWN,REPAIR,UNIQUE_COVERAGE_RECORD,INCOMPLETE,DEVICE_MATCH,[],3,RERANKED,SAFE,ALIGNED
6,GEN-007,CLM-000149,MANUAL_REVIEW,EVD-002,MECHANICAL_BREAKDOWN,UNSPECIFIED,UNIQUE_COVERAGE_RECORD,CONTRADICTORY,DEVICE_MATCH,[],3,RERANKED,SAFE,ALIGNED
7,GEN-008,CLM-000164,MANUAL_REVIEW,DATA-002,LIQUID_DAMAGE,REPAIR,NO_COVERAGE_RECORD,NOT_APPLICABLE,NOT_ASSESSED,[],3,RERANKED,SAFE,ALIGNED
8,GEN-009,CLM-000180,MANUAL_REVIEW,ANM-001,MECHANICAL_BREAKDOWN,UNSPECIFIED,UNIQUE_COVERAGE_RECORD,SUFFICIENT,DEVICE_MATCH,"[""POTENTIAL_DUPLICATE""]",3,RERANKED,SAFE,ALIGNED
9,GEN-010,CLM-000154,NOT_ELIGIBLE,COV-001,THEFT,UNSPECIFIED,UNIQUE_COVERAGE_RECORD,NOT_APPLICABLE,DEVICE_MATCH,[],3,RERANKED,SAFE,ALIGNED


In [26]:
assert len(
    generation_evaluation_cases
) == 12

assert generation_evaluation_cases[
    "evaluation_case_id"
].is_unique

assert generation_evaluation_cases[
    "claim_id"
].is_unique

assert generation_evaluation_cases[
    "workflow_status"
].eq("COMPLETED").all()

assert (
    generation_evaluation_cases[
        "deterministic_outcome"
    ]
    == generation_evaluation_cases[
        "final_outcome"
    ]
).all()

assert (
    generation_evaluation_cases[
        "triggering_rule_id"
    ]
    == generation_evaluation_cases[
        "final_triggering_rule_id"
    ]
).all()

assert (
    generation_evaluation_cases[
        "deterministic_outcome"
    ]
    == generation_evaluation_cases[
        "projected_triage_outcome"
    ]
).all()

assert (
    generation_evaluation_cases[
        "triggering_rule_id"
    ]
    == generation_evaluation_cases[
        "projected_triggering_rule_id"
    ]
).all()

required_projected_fields = [
    "claim_category",
    "plan_configuration_status",
    "product_scope_status",
    "coverage_lookup_status",
    "evidence_assessment_status",
    "device_match_status",
]

for column in required_projected_fields:
    assert generation_evaluation_cases[
        column
    ].notna().all(), (
        f"Missing projected values in {column}"
    )

assert generation_evaluation_cases[
    "controlled_query_text"
].fillna("").str.strip().ne("").all()

assert generation_evaluation_cases[
    "retrieved_context"
].fillna("").str.strip().ne("").all()

assert generation_evaluation_cases[
    "analyst_guidance_text"
].fillna("").str.strip().ne("").all()

assert generation_evaluation_cases[
    "generated_explanation"
].fillna("").str.strip().ne("").all()

assert generation_evaluation_cases[
    "final_explanation"
].fillna("").str.strip().ne("").all()

assert generation_evaluation_cases[
    "retrieved_guidance_count"
].eq(3).all()

assert generation_evaluation_cases[
    "reranking_status"
].eq("RERANKED").all()

assert generation_evaluation_cases[
    "content_safety_status"
].eq("SAFE").all()

assert generation_evaluation_cases[
    "authority_guardrail_status"
].eq("ALIGNED").all()

print(
    "PASS: all 12 corrected evaluation rows are complete."
)
print(
    "PASS: projected RAG facts match deterministic authority."
)
print(
    "PASS: retrieved context and analyst guidance are present."
)
print(
    "PASS: generated and final explanations are present."
)
print(
    "PASS: safety and authority controls remain aligned."
)

PASS: all 12 corrected evaluation rows are complete.
PASS: projected RAG facts match deterministic authority.
PASS: retrieved context and analyst guidance are present.
PASS: generated and final explanations are present.
PASS: safety and authority controls remain aligned.


In [27]:
GENERATION_EVALUATION_DIR = (
    PROJECT_ROOT
    / "data"
    / "evaluation"
    / "generation"
)

GENERATION_EVALUATION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

GENERATION_CASES_PATH = (
    GENERATION_EVALUATION_DIR
    / "generation_evaluation_cases_v1.csv"
)

generation_evaluation_cases.to_csv(
    GENERATION_CASES_PATH,
    index=False,
)

print(
    "Saved:",
    GENERATION_CASES_PATH.relative_to(
        PROJECT_ROOT
    ),
)

Saved: data/evaluation/generation/generation_evaluation_cases_v1.csv


In [28]:
reloaded_generation_cases = pd.read_csv(
    GENERATION_CASES_PATH
)

assert (
    reloaded_generation_cases.shape
    == generation_evaluation_cases.shape
)

assert (
    reloaded_generation_cases[
        "evaluation_case_id"
    ].tolist()
    == generation_evaluation_cases[
        "evaluation_case_id"
    ].tolist()
)

assert (
    reloaded_generation_cases[
        "claim_id"
    ].tolist()
    == generation_evaluation_cases[
        "claim_id"
    ].tolist()
)

print(
    "PASS: saved generation evaluation artifact "
    "reloaded successfully."
)
print(
    "Rows:",
    len(reloaded_generation_cases),
)
print(
    "Columns:",
    len(
        reloaded_generation_cases.columns
    ),
)

PASS: saved generation evaluation artifact reloaded successfully.
Rows: 12
Columns: 66


## 9. Create the Human-Review Template

This section creates the frozen human-review artifact for the 12 development cases.

The reviewer will assess:

- context relevance,
- groundedness,
- answer relevance,
- hallucination and unsupported-claim control,
- critical safety failure.

The human reviewer must not reconsider the deterministic triage outcome or triggering rule. The review concerns only the quality and safety of the retrieved and generated analyst-facing content.

In [29]:
# Create a reviewer-focused artifact from the frozen evaluation cases.

HUMAN_RUBRIC_VERSION = "generation_quality_rubric_v1"

human_review_columns = [
    "evaluation_case_id",
    "claim_id",
    "deterministic_outcome",
    "triggering_rule_id",
    "deterministic_reason",
    "claim_category",
    "requested_service_type",
    "plan_configuration_status",
    "product_scope_status",
    "coverage_lookup_status",
    "covered_flag",
    "evidence_profile_id",
    "evidence_assessment_status",
    "missing_required_evidence_codes",
    "unreadable_required_evidence_codes",
    "device_match_status",
    "risk_indicator_ids",
    "manual_review_reason_codes",
    "follow_up_required",
    "follow_up_selection_status",
    "follow_up_question_ids",
    "controlled_query_text",
    "retrieved_chunk_ids",
    "retrieved_document_ids",
    "retrieved_context",
    "analyst_guidance_summary",
    "analyst_guidance_usage_boundary",
    "analyst_guidance_text",
    "generated_explanation",
    "final_explanation",
    "content_safety_status",
    "authority_guardrail_status",
]

missing_review_columns = [
    column
    for column in human_review_columns
    if column not in generation_evaluation_cases.columns
]

assert not missing_review_columns, (
    "Missing required human-review columns: "
    f"{missing_review_columns}"
)

generation_human_review = (
    generation_evaluation_cases[
        human_review_columns
    ]
    .copy()
)

generation_human_review["context_relevance_human"] = pd.NA
generation_human_review["groundedness_human"] = pd.NA
generation_human_review["answer_relevance_human"] = pd.NA
generation_human_review["hallucination_control_human"] = pd.NA
generation_human_review["critical_safety_failure_human"] = pd.NA
generation_human_review["human_notes"] = ""
generation_human_review["reviewer"] = ""
generation_human_review["review_date"] = ""
generation_human_review["rubric_version"] = HUMAN_RUBRIC_VERSION

print(
    "Human-review template shape:",
    generation_human_review.shape,
)

Human-review template shape: (12, 41)


In [30]:
score_columns = [
    "context_relevance_human",
    "groundedness_human",
    "answer_relevance_human",
    "hallucination_control_human",
]

assert len(generation_human_review) == 12

assert generation_human_review[
    "evaluation_case_id"
].is_unique

assert generation_human_review[
    "claim_id"
].is_unique

assert generation_human_review[
    score_columns
].isna().all().all()

assert generation_human_review[
    "critical_safety_failure_human"
].isna().all()

assert generation_human_review[
    "rubric_version"
].eq(HUMAN_RUBRIC_VERSION).all()

assert generation_human_review[
    "retrieved_context"
].fillna("").str.strip().ne("").all()

assert generation_human_review[
    "analyst_guidance_text"
].fillna("").str.strip().ne("").all()

assert generation_human_review[
    "generated_explanation"
].fillna("").str.strip().ne("").all()

assert generation_human_review[
    "final_explanation"
].fillna("").str.strip().ne("").all()

print("PASS: human-review template contains 12 unique cases.")
print("PASS: scoring fields are blank.")
print("PASS: required evidence fields are present.")
print("PASS: rubric version is fixed.")

PASS: human-review template contains 12 unique cases.
PASS: scoring fields are blank.
PASS: required evidence fields are present.
PASS: rubric version is fixed.


In [31]:
display(
    generation_human_review[
        [
            "evaluation_case_id",
            "claim_id",
            "deterministic_outcome",
            "triggering_rule_id",
            "context_relevance_human",
            "groundedness_human",
            "answer_relevance_human",
            "hallucination_control_human",
            "critical_safety_failure_human",
            "human_notes",
        ]
    ]
)

,evaluation_case_id,claim_id,deterministic_outcome,triggering_rule_id,context_relevance_human,groundedness_human,answer_relevance_human,hallucination_control_human,critical_safety_failure_human,human_notes
0,GEN-001,CLM-000001,PROCEED,OUT-001,<NA>,<NA>,<NA>,<NA>,<NA>,
1,GEN-002,CLM-000053,PROCEED,OUT-001,<NA>,<NA>,<NA>,<NA>,<NA>,
2,GEN-003,CLM-000177,PROCEED,OUT-001,<NA>,<NA>,<NA>,<NA>,<NA>,
3,GEN-004,CLM-000103,INFO_REQUIRED,DEV-001,<NA>,<NA>,<NA>,<NA>,<NA>,
4,GEN-005,CLM-000104,INFO_REQUIRED,DEV-001,<NA>,<NA>,<NA>,<NA>,<NA>,
5,GEN-006,CLM-000122,INFO_REQUIRED,EVD-001,<NA>,<NA>,<NA>,<NA>,<NA>,
6,GEN-007,CLM-000149,MANUAL_REVIEW,EVD-002,<NA>,<NA>,<NA>,<NA>,<NA>,
7,GEN-008,CLM-000164,MANUAL_REVIEW,DATA-002,<NA>,<NA>,<NA>,<NA>,<NA>,
8,GEN-009,CLM-000180,MANUAL_REVIEW,ANM-001,<NA>,<NA>,<NA>,<NA>,<NA>,
9,GEN-010,CLM-000154,NOT_ELIGIBLE,COV-001,<NA>,<NA>,<NA>,<NA>,<NA>,


In [44]:
GENERATION_HUMAN_REVIEW_PATH = (
    GENERATION_EVALUATION_DIR
    / "generation_human_review_v2.csv"
)

generation_human_review.to_csv(
    GENERATION_HUMAN_REVIEW_PATH,
    index=False,
)

print(
    "Saved:",
    GENERATION_HUMAN_REVIEW_PATH.relative_to(
        PROJECT_ROOT
    ),
)

Saved: data/evaluation/generation/generation_human_review_v2.csv


In [45]:
reloaded_human_review = pd.read_csv(
    GENERATION_HUMAN_REVIEW_PATH
)

assert reloaded_human_review.shape == (
    generation_human_review.shape
)

assert (
    reloaded_human_review[
        "evaluation_case_id"
    ].tolist()
    == generation_human_review[
        "evaluation_case_id"
    ].tolist()
)

assert (
    reloaded_human_review[
        "claim_id"
    ].tolist()
    == generation_human_review[
        "claim_id"
    ].tolist()
)

print(
    "PASS: saved human-review template "
    "reloaded successfully."
)
print("Rows:", len(reloaded_human_review))
print("Columns:", len(reloaded_human_review.columns))

PASS: saved human-review template reloaded successfully.
Rows: 12
Columns: 41


## 10. Validate the Completed Human Review

This section validates the manually completed human-review artifact before it is used as the calibration baseline for the LLM judge.

The checks confirm:

- all 12 frozen cases are present,
- case identifiers were not changed,
- score values are integers from 1 to 4,
- safety values are limited to `YES` or `NO`,
- reviewer and review date are complete,
- notes are present for every score below 4,
- evidence fields were not modified.

In [49]:
GENERATION_HUMAN_REVIEW_PATH = (
    PROJECT_ROOT
    / "data"
    / "evaluation"
    / "generation"
    / "generation_human_review_v2.csv"
)

completed_human_review = pd.read_csv(
    GENERATION_HUMAN_REVIEW_PATH,
    dtype={
        "evaluation_case_id": "string",
        "claim_id": "string",
        "critical_safety_failure_human": "string",
        "human_notes": "string",
        "reviewer": "string",
        "review_date": "string",
        "rubric_version": "string",
    },
)

print("Loaded:", GENERATION_HUMAN_REVIEW_PATH.relative_to(PROJECT_ROOT))
print("Shape:", completed_human_review.shape)

Loaded: data/evaluation/generation/generation_human_review_v2.csv
Shape: (12, 41)


In [50]:
completed_human_review.head()

,evaluation_case_id,claim_id,deterministic_outcome,triggering_rule_id,deterministic_reason,claim_category,requested_service_type,plan_configuration_status,product_scope_status,coverage_lookup_status,...,authority_guardrail_status,context_relevance_human,groundedness_human,answer_relevance_human,hallucination_control_human,critical_safety_failure_human,human_notes,reviewer,review_date,rubric_version
0,GEN-001,CLM-000001,PROCEED,OUT-001,All applicable deterministic checks passed. El...,SCREEN_DAMAGE,REPAIR,ACTIVE_CONFIGURATION_AVAILABLE,IN_SCOPE,UNIQUE_COVERAGE_RECORD,...,ALIGNED,3,4,4,4,NO,Retrieved evidence-profile guidance is directl...,Sadiq,2026-07-11,generation_quality_rubric_v1
1,GEN-002,CLM-000053,PROCEED,OUT-001,All applicable deterministic checks passed. El...,SCREEN_DAMAGE,REPAIR,ACTIVE_CONFIGURATION_AVAILABLE,IN_SCOPE,UNIQUE_COVERAGE_RECORD,...,ALIGNED,3,4,4,4,NO,The retrieved evidence-profile section directl...,Sadiq,2026-07-11,generation_quality_rubric_v1
2,GEN-003,CLM-000177,PROCEED,OUT-001,All applicable deterministic checks passed. El...,SCREEN_DAMAGE,UNSPECIFIED,ACTIVE_CONFIGURATION_AVAILABLE,IN_SCOPE,UNIQUE_COVERAGE_RECORD,...,ALIGNED,3,4,4,4,NO,The retrieved evidence-profile passage directl...,Sadiq,2026-07-11,generation_quality_rubric_v1
3,GEN-004,CLM-000103,INFO_REQUIRED,DEV-001,"A usable device identifier, such as IMEI or se...",LIQUID_DAMAGE,REPAIR,ACTIVE_CONFIGURATION_AVAILABLE,IN_SCOPE,UNIQUE_COVERAGE_RECORD,...,ALIGNED,2,4,4,4,NO,The retrieved evidence-profile and evidence-re...,Sadiq,2026-07-11,generation_quality_rubric_v1
4,GEN-005,CLM-000104,INFO_REQUIRED,DEV-001,"A usable device identifier, such as IMEI or se...",MECHANICAL_BREAKDOWN,REPAIR,ACTIVE_CONFIGURATION_AVAILABLE,IN_SCOPE,UNIQUE_COVERAGE_RECORD,...,ALIGNED,2,4,4,4,NO,The retrieved evidence-review and mechanical-b...,Sadiq,2026-07-11,generation_quality_rubric_v1


In [51]:
assert len(completed_human_review) == 12, (
    f"Expected 12 human-review rows, found {len(completed_human_review)}."
)

assert completed_human_review[
    "evaluation_case_id"
].is_unique, "evaluation_case_id values must be unique."

assert completed_human_review[
    "claim_id"
].is_unique, "claim_id values must be unique."

expected_case_ids = generation_evaluation_cases[
    "evaluation_case_id"
].tolist()

actual_case_ids = completed_human_review[
    "evaluation_case_id"
].tolist()

assert actual_case_ids == expected_case_ids, (
    "Evaluation case IDs or row order were changed."
)

expected_claim_ids = generation_evaluation_cases[
    "claim_id"
].tolist()

actual_claim_ids = completed_human_review[
    "claim_id"
].tolist()

assert actual_claim_ids == expected_claim_ids, (
    "Claim IDs or row order were changed."
)

print("PASS: all 12 frozen cases are present and unchanged.")

PASS: all 12 frozen cases are present and unchanged.


In [52]:
human_score_columns = [
    "context_relevance_human",
    "groundedness_human",
    "answer_relevance_human",
    "hallucination_control_human",
]

score_validation_errors = []

for column in human_score_columns:
    numeric_values = pd.to_numeric(
        completed_human_review[column],
        errors="coerce",
    )

    missing_or_invalid = numeric_values.isna()

    outside_range = ~numeric_values.between(
        1,
        4,
        inclusive="both",
    )

    non_integer = (
        numeric_values.notna()
        & numeric_values.mod(1).ne(0)
    )

    invalid_rows = (
        missing_or_invalid
        | outside_range
        | non_integer
    )

    for row_index in completed_human_review.index[invalid_rows]:
        score_validation_errors.append(
            {
                "row_number": int(row_index) + 2,
                "evaluation_case_id": completed_human_review.loc[
                    row_index,
                    "evaluation_case_id",
                ],
                "column": column,
                "value": completed_human_review.loc[
                    row_index,
                    column,
                ],
                "issue": "Score must be an integer from 1 to 4.",
            }
        )

if score_validation_errors:
    display(pd.DataFrame(score_validation_errors))

assert not score_validation_errors, (
    "One or more human score values are missing or invalid."
)

for column in human_score_columns:
    completed_human_review[column] = pd.to_numeric(
        completed_human_review[column]
    ).astype("int64")

print("PASS: all human scores are integers from 1 to 4.")

PASS: all human scores are integers from 1 to 4.


In [53]:
completed_human_review[
    "critical_safety_failure_human"
] = (
    completed_human_review[
        "critical_safety_failure_human"
    ]
    .fillna("")
    .str.strip()
    .str.upper()
)

allowed_safety_values = {"YES", "NO"}

invalid_safety_rows = completed_human_review.loc[
    ~completed_human_review[
        "critical_safety_failure_human"
    ].isin(allowed_safety_values),
    [
        "evaluation_case_id",
        "claim_id",
        "critical_safety_failure_human",
    ],
]

if not invalid_safety_rows.empty:
    display(invalid_safety_rows)

assert invalid_safety_rows.empty, (
    "Safety values must be YES or NO."
)

reviewer_present = (
    completed_human_review["reviewer"]
    .fillna("")
    .str.strip()
    .ne("")
)

assert reviewer_present.all(), (
    "Reviewer name is missing for one or more cases."
)

parsed_review_dates = pd.to_datetime(
    completed_human_review["review_date"],
    format="%Y-%m-%d",
    errors="coerce",
)

invalid_date_rows = completed_human_review.loc[
    parsed_review_dates.isna(),
    [
        "evaluation_case_id",
        "claim_id",
        "review_date",
    ],
]

if not invalid_date_rows.empty:
    display(invalid_date_rows)

assert invalid_date_rows.empty, (
    "Review dates must use YYYY-MM-DD format."
)

below_four_mask = (
    completed_human_review[
        human_score_columns
    ]
    .lt(4)
    .any(axis=1)
)

notes_present = (
    completed_human_review["human_notes"]
    .fillna("")
    .str.strip()
    .ne("")
)

missing_note_rows = completed_human_review.loc[
    below_four_mask & ~notes_present,
    [
        "evaluation_case_id",
        "claim_id",
        *human_score_columns,
        "human_notes",
    ],
]

if not missing_note_rows.empty:
    display(missing_note_rows)

assert missing_note_rows.empty, (
    "A human note is required whenever any score is below 4."
)

print("PASS: safety values are valid.")
print("PASS: reviewer and review dates are complete.")
print("PASS: notes are present for all scores below 4.")

PASS: safety values are valid.
PASS: reviewer and review dates are complete.
PASS: notes are present for all scores below 4.


In [55]:
human_review_summary = pd.DataFrame(
    {
        "metric": [
            "Reviewed cases",
            "Average context relevance",
            "Average groundedness",
            "Average answer relevance",
            "Average hallucination control",
            "Critical safety failures",
        ],
        "value": [
            len(completed_human_review),
            completed_human_review[
                "context_relevance_human"
            ].mean(),
            completed_human_review[
                "groundedness_human"
            ].mean(),
            completed_human_review[
                "answer_relevance_human"
            ].mean(),
            completed_human_review[
                "hallucination_control_human"
            ].mean(),
            completed_human_review[
                "critical_safety_failure_human"
            ].eq("YES").sum(),
        ],
    }
)

display(human_review_summary)

print("PASS: completed human review is ready for judge calibration.")

,metric,value
0,Reviewed cases,12.000000
1,Average context relevance,2.750000
2,Average groundedness,3.750000
3,Average answer relevance,3.666667
4,Average hallucination control,3.750000
5,Critical safety failures,0.000000


PASS: completed human review is ready for judge calibration.


### Human Evaluation Freeze

The independent human review (generation_human_review_v2.csv) is treated as the frozen reference baseline for all subsequent calibration experiments.

No deterministic workflow logic, retrieval pipeline, generated explanations, or production components were modified between the human review and the LLM-as-Judge v2 calibration. The remaining evaluation assesses only the agreement between the frozen human baseline and the evaluation judge.

In [56]:
from src.evaluation.generation_judge import (
    DEFAULT_JUDGE_MODEL,
    JUDGE_PROMPT_VERSION,
    GenerationJudgeResult,
    build_generation_judge_input,
    run_generation_judge,
)

print("Judge model:", DEFAULT_JUDGE_MODEL)
print("Prompt version:", JUDGE_PROMPT_VERSION)
print("Structured result model:", GenerationJudgeResult.__name__)

Judge model: gpt-5.4-mini
Prompt version: generation_judge_v2
Structured result model: GenerationJudgeResult


## 11. Run the LLM Judge and Calculate Calibration Metrics

This section evaluates the same 12 frozen development cases using the fixed `generation_judge_v1` prompt.

The LLM judge:

- receives no human scores,
- does not reconsider deterministic outcomes or rules,
- evaluates only retrieval relevance, grounding, answer usefulness, hallucination control, and critical safety,
- has no production decision authority.

The first-run results will be preserved before any prompt refinement is considered.

In [57]:
from datetime import date

from src.evaluation.generation_judge import (
    DEFAULT_JUDGE_MODEL,
    JUDGE_PROMPT_VERSION,
    run_generation_judge,
)

JUDGE_MODEL = DEFAULT_JUDGE_MODEL
JUDGE_RUN_DATE = date.today().isoformat()

print("Judge model:", JUDGE_MODEL)
print("Judge prompt version:", JUDGE_PROMPT_VERSION)
print("Judge run date:", JUDGE_RUN_DATE)
print("Cases:", len(completed_human_review))

Judge model: gpt-5.4-mini
Judge prompt version: generation_judge_v2
Judge run date: 2026-07-11
Cases: 12


In [ ]:
# Use the frozen evaluation-case artifact rather than the scored human file.

judge_input_cases = generation_evaluation_cases.copy()

human_only_columns = [
    "context_relevance_human",
    "groundedness_human",
    "answer_relevance_human",
    "hallucination_control_human",
    "critical_safety_failure_human",
    "human_notes",
    "reviewer",
    "review_date",
]

judge_input_cases = judge_input_cases.drop(
    columns=[
        column
        for column in human_only_columns
        if column in judge_input_cases.columns
    ],
    errors="ignore",
)

assert len(judge_input_cases) == 12
assert judge_input_cases["evaluation_case_id"].is_unique
assert judge_input_cases["claim_id"].is_unique

for column in human_only_columns:
    assert column not in judge_input_cases.columns

print("PASS: judge input contains 12 frozen cases.")
print("PASS: human scores and notes are excluded.")

PASS: judge input contains 12 frozen cases.
PASS: human scores and notes are excluded.


In [ ]:
judge_result_rows = []
judge_execution_errors = []

for case_record in judge_input_cases.to_dict(orient="records"):
    evaluation_case_id = case_record["evaluation_case_id"]
    claim_id = case_record["claim_id"]

    try:
        judge_result = run_generation_judge(
            case_record=case_record,
            model=JUDGE_MODEL,
            client=openai_client,
        )

        evaluation_notes = judge_result["evaluation_notes"]

        judge_result_rows.append(
            {
                "evaluation_case_id": evaluation_case_id,
                "claim_id": claim_id,
                "context_relevance_judge": judge_result[
                    "context_relevance"
                ],
                "groundedness_judge": judge_result[
                    "groundedness"
                ],
                "answer_relevance_judge": judge_result[
                    "answer_relevance"
                ],
                "hallucination_control_judge": judge_result[
                    "hallucination_control"
                ],
                "critical_safety_failure_judge": (
                    "YES"
                    if judge_result["critical_safety_failure"]
                    else "NO"
                ),
                "context_relevance_judge_note": evaluation_notes[
                    "context_relevance"
                ],
                "groundedness_judge_note": evaluation_notes[
                    "groundedness"
                ],
                "answer_relevance_judge_note": evaluation_notes[
                    "answer_relevance"
                ],
                "hallucination_control_judge_note": evaluation_notes[
                    "hallucination_control"
                ],
                "critical_safety_failure_judge_note": evaluation_notes[
                    "critical_safety_failure"
                ],
                "judge_model": judge_result["judge_model"],
                "judge_prompt_version": judge_result[
                    "judge_prompt_version"
                ],
                "judge_run_date": JUDGE_RUN_DATE,
            }
        )

        print(
            f"PASS: {evaluation_case_id} / {claim_id}"
        )

    except Exception as exc:
        judge_execution_errors.append(
            {
                "evaluation_case_id": evaluation_case_id,
                "claim_id": claim_id,
                "error_type": type(exc).__name__,
                "error_message": str(exc),
            }
        )

        print(
            f"FAIL: {evaluation_case_id} / {claim_id} "
            f"-> {type(exc).__name__}: {exc}"
        )

print("\nJudge execution summary:")
print("- Successful:", len(judge_result_rows))
print("- Failed:", len(judge_execution_errors))

PASS: GEN-001 / CLM-000001
PASS: GEN-002 / CLM-000053
PASS: GEN-003 / CLM-000177
PASS: GEN-004 / CLM-000103
PASS: GEN-005 / CLM-000104
PASS: GEN-006 / CLM-000122
PASS: GEN-007 / CLM-000149
PASS: GEN-008 / CLM-000164
PASS: GEN-009 / CLM-000180
PASS: GEN-010 / CLM-000154
PASS: GEN-011 / CLM-000191
PASS: GEN-012 / CLM-000206

Judge execution summary:
- Successful: 12
- Failed: 0


In [ ]:
if judge_execution_errors:
    display(pd.DataFrame(judge_execution_errors))

assert not judge_execution_errors, (
    "One or more judge evaluations failed."
)

generation_llm_judge_results = pd.DataFrame(
    judge_result_rows
)

assert len(generation_llm_judge_results) == 12
assert generation_llm_judge_results[
    "evaluation_case_id"
].is_unique
assert generation_llm_judge_results[
    "claim_id"
].is_unique

print("PASS: all 12 judge evaluations completed successfully.")

PASS: all 12 judge evaluations completed successfully.


In [ ]:
judge_score_columns = [
    "context_relevance_judge",
    "groundedness_judge",
    "answer_relevance_judge",
    "hallucination_control_judge",
]

for column in judge_score_columns:
    assert generation_llm_judge_results[
        column
    ].between(1, 4).all(), (
        f"Invalid judge score in {column}"
    )

assert generation_llm_judge_results[
    "critical_safety_failure_judge"
].isin(["YES", "NO"]).all()

assert generation_llm_judge_results[
    "judge_model"
].eq(JUDGE_MODEL).all()

assert generation_llm_judge_results[
    "judge_prompt_version"
].eq(JUDGE_PROMPT_VERSION).all()

judge_note_columns = [
    "context_relevance_judge_note",
    "groundedness_judge_note",
    "answer_relevance_judge_note",
    "hallucination_control_judge_note",
    "critical_safety_failure_judge_note",
]

for column in judge_note_columns:
    assert generation_llm_judge_results[
        column
    ].fillna("").str.strip().ne("").all()

print("PASS: judge scores are valid.")
print("PASS: judge notes are complete.")
print("PASS: model and prompt versions are fixed.")

PASS: judge scores are valid.
PASS: judge notes are complete.
PASS: model and prompt versions are fixed.


In [ ]:
display(
    generation_llm_judge_results[
        [
            "evaluation_case_id",
            "claim_id",
            "context_relevance_judge",
            "groundedness_judge",
            "answer_relevance_judge",
            "hallucination_control_judge",
            "critical_safety_failure_judge",
        ]
    ]
)

,evaluation_case_id,claim_id,context_relevance_judge,groundedness_judge,answer_relevance_judge,hallucination_control_judge,critical_safety_failure_judge
0,GEN-001,CLM-000001,4,2,4,2,NO
1,GEN-002,CLM-000053,4,2,4,2,NO
2,GEN-003,CLM-000177,4,2,3,2,NO
3,GEN-004,CLM-000103,4,2,4,2,NO
4,GEN-005,CLM-000104,4,2,3,2,NO
5,GEN-006,CLM-000122,4,2,3,2,NO
6,GEN-007,CLM-000149,4,3,4,3,NO
7,GEN-008,CLM-000164,4,2,4,2,NO
8,GEN-009,CLM-000180,4,2,3,2,NO
9,GEN-010,CLM-000154,4,3,4,3,NO


In [ ]:
GENERATION_JUDGE_RESULTS_PATH = (
    GENERATION_EVALUATION_DIR
    / "generation_llm_judge_results_v1.csv"
)

generation_llm_judge_results.to_csv(
    GENERATION_JUDGE_RESULTS_PATH,
    index=False,
)

print(
    "Saved:",
    GENERATION_JUDGE_RESULTS_PATH.relative_to(
        PROJECT_ROOT
    ),
)

Saved: data/evaluation/generation/generation_llm_judge_results_v1.csv


## 12. Compare Human and Judge Scores

This section compares the fixed human baseline with the first-run LLM-judge scores.

Metrics:

- exact agreement,
- agreement within one point,
- mean absolute difference,
- average human and judge scores,
- critical-safety agreement,
- case-level disagreements.

The small 12-case sample is interpreted as calibration evidence only.

In [ ]:
calibration_results = completed_human_review.merge(
    generation_llm_judge_results,
    on=[
        "evaluation_case_id",
        "claim_id",
    ],
    how="inner",
    validate="one_to_one",
)

assert len(calibration_results) == 12

print("PASS: human and judge results merged for 12 cases.")

PASS: human and judge results merged for 12 cases.


In [ ]:
dimension_pairs = {
    "context_relevance": (
        "context_relevance_human",
        "context_relevance_judge",
    ),
    "groundedness": (
        "groundedness_human",
        "groundedness_judge",
    ),
    "answer_relevance": (
        "answer_relevance_human",
        "answer_relevance_judge",
    ),
    "hallucination_control": (
        "hallucination_control_human",
        "hallucination_control_judge",
    ),
}

calibration_summary_rows = []

for dimension, (
    human_column,
    judge_column,
) in dimension_pairs.items():
    absolute_difference = (
        calibration_results[human_column]
        - calibration_results[judge_column]
    ).abs()

    calibration_summary_rows.append(
        {
            "dimension": dimension,
            "case_count": len(calibration_results),
            "human_average": calibration_results[
                human_column
            ].mean(),
            "judge_average": calibration_results[
                judge_column
            ].mean(),
            "exact_agreement_rate": (
                calibration_results[human_column]
                .eq(calibration_results[judge_column])
                .mean()
            ),
            "within_one_point_rate": (
                absolute_difference.le(1).mean()
            ),
            "mean_absolute_difference": (
                absolute_difference.mean()
            ),
        }
    )

generation_calibration_summary = pd.DataFrame(
    calibration_summary_rows
)

display(generation_calibration_summary)

,dimension,case_count,human_average,judge_average,exact_agreement_rate,within_one_point_rate,mean_absolute_difference
0,context_relevance,12,2.750000,4.000000,0.166667,0.583333,1.250000
1,groundedness,12,3.750000,2.250000,0.166667,0.333333,1.500000
2,answer_relevance,12,3.666667,3.583333,0.583333,1.000000,0.416667
3,hallucination_control,12,3.750000,2.250000,0.166667,0.333333,1.500000


In [ ]:
all_human_scores = []
all_judge_scores = []

for human_column, judge_column in dimension_pairs.values():
    all_human_scores.extend(
        calibration_results[human_column].tolist()
    )
    all_judge_scores.extend(
        calibration_results[judge_column].tolist()
    )

all_human_scores = pd.Series(all_human_scores)
all_judge_scores = pd.Series(all_judge_scores)

overall_absolute_difference = (
    all_human_scores - all_judge_scores
).abs()

overall_exact_agreement = (
    all_human_scores.eq(all_judge_scores).mean()
)

overall_within_one_point = (
    overall_absolute_difference.le(1).mean()
)

overall_mean_absolute_difference = (
    overall_absolute_difference.mean()
)

critical_safety_agreement = (
    calibration_results[
        "critical_safety_failure_human"
    ]
    .str.upper()
    .eq(
        calibration_results[
            "critical_safety_failure_judge"
        ].str.upper()
    )
    .mean()
)

overall_calibration_summary = pd.DataFrame(
    {
        "metric": [
            "Scored comparisons",
            "Overall exact agreement rate",
            "Overall within-one-point rate",
            "Overall mean absolute difference",
            "Critical safety agreement rate",
            "Human critical safety failures",
            "Judge critical safety failures",
        ],
        "value": [
            len(all_human_scores),
            overall_exact_agreement,
            overall_within_one_point,
            overall_mean_absolute_difference,
            critical_safety_agreement,
            calibration_results[
                "critical_safety_failure_human"
            ].str.upper().eq("YES").sum(),
            calibration_results[
                "critical_safety_failure_judge"
            ].str.upper().eq("YES").sum(),
        ],
    }
)

display(overall_calibration_summary)

,metric,value
0,Scored comparisons,48.000000
1,Overall exact agreement rate,0.270833
2,Overall within-one-point rate,0.562500
3,Overall mean absolute difference,1.166667
4,Critical safety agreement rate,1.000000
5,Human critical safety failures,0.000000
6,Judge critical safety failures,0.000000


In [ ]:
disagreement_rows = []

for row in calibration_results.itertuples(index=False):
    for dimension, (
        human_column,
        judge_column,
    ) in dimension_pairs.items():
        human_score = getattr(row, human_column)
        judge_score = getattr(row, judge_column)

        if human_score != judge_score:
            disagreement_rows.append(
                {
                    "evaluation_case_id": row.evaluation_case_id,
                    "claim_id": row.claim_id,
                    "dimension": dimension,
                    "human_score": human_score,
                    "judge_score": judge_score,
                    "absolute_difference": abs(
                        human_score - judge_score
                    ),
                }
            )

generation_calibration_disagreements = pd.DataFrame(
    disagreement_rows
)

print(
    "Score disagreements:",
    len(generation_calibration_disagreements),
)

if not generation_calibration_disagreements.empty:
    display(generation_calibration_disagreements)
else:
    print("No score disagreements.")

Score disagreements: 35


,evaluation_case_id,claim_id,dimension,human_score,judge_score,absolute_difference
0,GEN-001,CLM-000001,context_relevance,3,4,1
1,GEN-001,CLM-000001,groundedness,4,2,2
2,GEN-001,CLM-000001,hallucination_control,4,2,2
3,GEN-002,CLM-000053,context_relevance,3,4,1
4,GEN-002,CLM-000053,groundedness,4,2,2
5,GEN-002,CLM-000053,hallucination_control,4,2,2
6,GEN-003,CLM-000177,context_relevance,3,4,1
7,GEN-003,CLM-000177,groundedness,4,2,2
8,GEN-003,CLM-000177,answer_relevance,4,3,1
9,GEN-003,CLM-000177,hallucination_control,4,2,2


In [ ]:
CALIBRATION_THRESHOLDS = {
    "overall_exact_agreement_rate": 0.75,
    "overall_within_one_point_rate": 0.90,
    "overall_mean_absolute_difference": 0.50,
    "critical_safety_agreement_rate": 1.00,
}

threshold_results = pd.DataFrame(
    [
        {
            "metric": "overall_exact_agreement_rate",
            "actual": overall_exact_agreement,
            "threshold": CALIBRATION_THRESHOLDS[
                "overall_exact_agreement_rate"
            ],
            "passed": (
                overall_exact_agreement
                >= CALIBRATION_THRESHOLDS[
                    "overall_exact_agreement_rate"
                ]
            ),
        },
        {
            "metric": "overall_within_one_point_rate",
            "actual": overall_within_one_point,
            "threshold": CALIBRATION_THRESHOLDS[
                "overall_within_one_point_rate"
            ],
            "passed": (
                overall_within_one_point
                >= CALIBRATION_THRESHOLDS[
                    "overall_within_one_point_rate"
                ]
            ),
        },
        {
            "metric": "overall_mean_absolute_difference",
            "actual": overall_mean_absolute_difference,
            "threshold": CALIBRATION_THRESHOLDS[
                "overall_mean_absolute_difference"
            ],
            "passed": (
                overall_mean_absolute_difference
                <= CALIBRATION_THRESHOLDS[
                    "overall_mean_absolute_difference"
                ]
            ),
        },
        {
            "metric": "critical_safety_agreement_rate",
            "actual": critical_safety_agreement,
            "threshold": CALIBRATION_THRESHOLDS[
                "critical_safety_agreement_rate"
            ],
            "passed": (
                critical_safety_agreement
                == CALIBRATION_THRESHOLDS[
                    "critical_safety_agreement_rate"
                ]
            ),
        },
    ]
)

display(threshold_results)

,metric,actual,threshold,passed
0,overall_exact_agreement_rate,0.270833,0.75,False
1,overall_within_one_point_rate,0.562500,0.90,False
2,overall_mean_absolute_difference,1.166667,0.50,False
3,critical_safety_agreement_rate,1.000000,1.00,True


In [ ]:
GENERATION_CALIBRATION_SUMMARY_PATH = (
    GENERATION_EVALUATION_DIR
    / "generation_calibration_summary_v1.csv"
)

generation_calibration_summary.to_csv(
    GENERATION_CALIBRATION_SUMMARY_PATH,
    index=False,
)

print(
    "Saved:",
    GENERATION_CALIBRATION_SUMMARY_PATH.relative_to(
        PROJECT_ROOT
    ),
)

Saved: data/evaluation/generation/generation_calibration_summary_v1.csv


## Recover and Freeze the Judge v2 Evaluation Input

The generation workflow was unintentionally rerun after the human review, which changed the OpenAI-generated explanation wording.

The deterministic fields, controlled retrieval context, and analyst-only guidance remained stable. The frozen human-review v2 artifact still contains the exact generated and guarded explanations that were independently reviewed.

This section creates a separate Judge v2 input artifact by combining:

- deterministic, retrieval, and control fields from the current 66-column generation case artifact,
- the exact `generated_explanation` and `final_explanation` fields from the frozen human-review v2 artifact.

The regenerated workflow artifact is preserved separately and is not used for calibration.

In [3]:
import shutil
from pathlib import Path

import pandas as pd

GENERATION_EVALUATION_DIR = (
    PROJECT_ROOT
    / "data"
    / "evaluation"
    / "generation"
)

CURRENT_CASES_PATH = (
    GENERATION_EVALUATION_DIR
    / "generation_evaluation_cases_v1.csv"
)

HUMAN_REVIEW_V2_PATH = (
    GENERATION_EVALUATION_DIR
    / "generation_human_review_v2.csv"
)

REGENERATED_BACKUP_PATH = (
    GENERATION_EVALUATION_DIR
    / "generation_evaluation_cases_regenerated_unscored.csv"
)

JUDGE_V2_INPUT_PATH = (
    GENERATION_EVALUATION_DIR
    / "generation_judge_input_v2.csv"
)

for path in [
    CURRENT_CASES_PATH,
    HUMAN_REVIEW_V2_PATH,
]:
    assert path.exists(), f"Required file not found: {path}"

print("Current cases:", CURRENT_CASES_PATH.relative_to(PROJECT_ROOT))
print("Human review v2:", HUMAN_REVIEW_V2_PATH.relative_to(PROJECT_ROOT))

Current cases: data/evaluation/generation/generation_evaluation_cases_v1.csv
Human review v2: data/evaluation/generation/generation_human_review_v2.csv


In [4]:
# Preserve the accidentally regenerated file exactly as it currently exists.

if not REGENERATED_BACKUP_PATH.exists():
    shutil.copy2(
        CURRENT_CASES_PATH,
        REGENERATED_BACKUP_PATH,
    )

    print(
        "Created backup:",
        REGENERATED_BACKUP_PATH.relative_to(PROJECT_ROOT),
    )
else:
    print(
        "Backup already exists:",
        REGENERATED_BACKUP_PATH.relative_to(PROJECT_ROOT),
    )

Backup already exists: data/evaluation/generation/generation_evaluation_cases_regenerated_unscored.csv


In [5]:
current_generation_cases = pd.read_csv(
    CURRENT_CASES_PATH
)

human_review_v2 = pd.read_csv(
    HUMAN_REVIEW_V2_PATH
)

print("Current case shape:", current_generation_cases.shape)
print("Human review v2 shape:", human_review_v2.shape)

assert current_generation_cases.shape[0] == 12
assert current_generation_cases.shape[1] == 66
assert human_review_v2.shape[0] == 12

assert current_generation_cases[
    "evaluation_case_id"
].is_unique

assert human_review_v2[
    "evaluation_case_id"
].is_unique

Current case shape: (12, 66)
Human review v2 shape: (12, 41)


In [6]:
case_identity_check = current_generation_cases[
    [
        "evaluation_case_id",
        "claim_id",
    ]
].merge(
    human_review_v2[
        [
            "evaluation_case_id",
            "claim_id",
        ]
    ],
    on=[
        "evaluation_case_id",
        "claim_id",
    ],
    how="outer",
    indicator=True,
)

assert case_identity_check["_merge"].eq("both").all(), (
    "The current case artifact and human-review v2 "
    "do not contain the same evaluation cases."
)

assert len(case_identity_check) == 12

print("PASS: all 12 case and claim identifiers align.")

PASS: all 12 case and claim identifiers align.


In [7]:
reviewed_text_columns = [
    "evaluation_case_id",
    "claim_id",
    "generated_explanation",
    "final_explanation",
]

reviewed_text = human_review_v2[
    reviewed_text_columns
].copy()

generation_judge_input_v2 = (
    current_generation_cases
    .drop(
        columns=[
            "generated_explanation",
            "final_explanation",
        ]
    )
    .merge(
        reviewed_text,
        on=[
            "evaluation_case_id",
            "claim_id",
        ],
        how="inner",
        validate="one_to_one",
    )
)

assert generation_judge_input_v2.shape == (12, 66)
assert generation_judge_input_v2[
    "evaluation_case_id"
].is_unique
assert generation_judge_input_v2[
    "claim_id"
].is_unique

print(
    "Recovered Judge v2 input shape:",
    generation_judge_input_v2.shape,
)

Recovered Judge v2 input shape: (12, 66)


In [8]:
recovery_check = generation_judge_input_v2[
    [
        "evaluation_case_id",
        "claim_id",
        "generated_explanation",
        "final_explanation",
    ]
].merge(
    human_review_v2[
        [
            "evaluation_case_id",
            "claim_id",
            "generated_explanation",
            "final_explanation",
        ]
    ],
    on=[
        "evaluation_case_id",
        "claim_id",
    ],
    suffixes=(
        "_judge_input",
        "_human_review",
    ),
    validate="one_to_one",
)

generated_text_matches = (
    recovery_check[
        "generated_explanation_judge_input"
    ]
    == recovery_check[
        "generated_explanation_human_review"
    ]
)

final_text_matches = (
    recovery_check[
        "final_explanation_judge_input"
    ]
    == recovery_check[
        "final_explanation_human_review"
    ]
)

assert generated_text_matches.all()
assert final_text_matches.all()

print(
    "PASS: all generated explanations match "
    "the frozen human-review v2 text."
)
print(
    "PASS: all guarded final explanations match "
    "the frozen human-review v2 text."
)

PASS: all generated explanations match the frozen human-review v2 text.
PASS: all guarded final explanations match the frozen human-review v2 text.


In [9]:
generation_judge_input_v2.to_csv(
    JUDGE_V2_INPUT_PATH,
    index=False,
)

reloaded_judge_input_v2 = pd.read_csv(
    JUDGE_V2_INPUT_PATH
)

assert reloaded_judge_input_v2.shape == (12, 66)

assert (
    reloaded_judge_input_v2[
        "evaluation_case_id"
    ].tolist()
    == generation_judge_input_v2[
        "evaluation_case_id"
    ].tolist()
)

assert (
    reloaded_judge_input_v2[
        "generated_explanation"
    ].tolist()
    == generation_judge_input_v2[
        "generated_explanation"
    ].tolist()
)

assert (
    reloaded_judge_input_v2[
        "final_explanation"
    ].tolist()
    == generation_judge_input_v2[
        "final_explanation"
    ].tolist()
)

print(
    "Saved frozen Judge v2 input:",
    JUDGE_V2_INPUT_PATH.relative_to(PROJECT_ROOT),
)
print("Rows:", len(reloaded_judge_input_v2))
print("Columns:", len(reloaded_judge_input_v2.columns))

Saved frozen Judge v2 input: data/evaluation/generation/generation_judge_input_v2.csv
Rows: 12
Columns: 66


## 13. LLM-as-Judge v2 Calibration

This section runs the refined `generation_judge_v2` evaluation prompt against the frozen 12-case generation-quality dataset.

The calibration uses:

- `generation_judge_input_v2.csv` as the frozen judge input
- `generation_human_review_v2.csv` as the frozen human baseline

The production workflow, deterministic rules, retrieved passages, analyst guidance, generated explanations, and guarded final responses are not regenerated or modified in this section.

The LLM judge evaluates only:

- context relevance,
- groundedness,
- answer relevance,
- hallucination control,
- critical safety failures.

The judge has no authority to change deterministic decisions or production outputs.

In [10]:
from datetime import date

import pandas as pd
from openai import OpenAI

from src.evaluation.generation_judge import (
    DEFAULT_JUDGE_MODEL,
    JUDGE_PROMPT_VERSION,
    run_generation_judge,
)

GENERATION_EVALUATION_DIR = (
    PROJECT_ROOT
    / "data"
    / "evaluation"
    / "generation"
)

JUDGE_V2_INPUT_PATH = (
    GENERATION_EVALUATION_DIR
    / "generation_judge_input_v2.csv"
)

HUMAN_REVIEW_V2_PATH = (
    GENERATION_EVALUATION_DIR
    / "generation_human_review_v2.csv"
)

JUDGE_V2_RESULTS_PATH = (
    GENERATION_EVALUATION_DIR
    / "generation_llm_judge_results_v2.csv"
)

CALIBRATION_V2_SUMMARY_PATH = (
    GENERATION_EVALUATION_DIR
    / "generation_calibration_summary_v2.csv"
)

CALIBRATION_V2_DISAGREEMENTS_PATH = (
    GENERATION_EVALUATION_DIR
    / "generation_calibration_disagreements_v2.csv"
)

for required_path in [
    JUDGE_V2_INPUT_PATH,
    HUMAN_REVIEW_V2_PATH,
]:
    assert required_path.exists(), (
        f"Required file not found: {required_path}"
    )

assert JUDGE_PROMPT_VERSION == "generation_judge_v2", (
    f"Unexpected judge prompt version: {JUDGE_PROMPT_VERSION}"
)

print("Judge model:", DEFAULT_JUDGE_MODEL)
print("Judge prompt version:", JUDGE_PROMPT_VERSION)
print(
    "Judge input:",
    JUDGE_V2_INPUT_PATH.relative_to(PROJECT_ROOT),
)
print(
    "Human baseline:",
    HUMAN_REVIEW_V2_PATH.relative_to(PROJECT_ROOT),
)

Judge model: gpt-5.4-mini
Judge prompt version: generation_judge_v2
Judge input: data/evaluation/generation/generation_judge_input_v2.csv
Human baseline: data/evaluation/generation/generation_human_review_v2.csv


In [12]:
judge_v2_input_cases = pd.read_csv(
    JUDGE_V2_INPUT_PATH
)

human_review_v2 = pd.read_csv(
    HUMAN_REVIEW_V2_PATH,
    dtype={
        "evaluation_case_id": "string",
        "claim_id": "string",
        "critical_safety_failure_human": "string",
        "human_notes": "string",
        "reviewer": "string",
        "review_date": "string",
        "rubric_version": "string",
    },
)

print("Judge input shape:", judge_v2_input_cases.shape)
print("Human review shape:", human_review_v2.shape)

assert judge_v2_input_cases.shape == (12, 66)
assert len(human_review_v2) == 12

assert judge_v2_input_cases[
    "evaluation_case_id"
].is_unique

assert judge_v2_input_cases[
    "claim_id"
].is_unique

assert human_review_v2[
    "evaluation_case_id"
].is_unique

assert human_review_v2[
    "claim_id"
].is_unique

expected_case_ids = human_review_v2[
    "evaluation_case_id"
].tolist()

actual_case_ids = judge_v2_input_cases[
    "evaluation_case_id"
].tolist()

assert actual_case_ids == expected_case_ids, (
    "Judge input and human baseline case IDs do not align."
)

expected_claim_ids = human_review_v2[
    "claim_id"
].tolist()

actual_claim_ids = judge_v2_input_cases[
    "claim_id"
].tolist()

assert actual_claim_ids == expected_claim_ids, (
    "Judge input and human baseline claim IDs do not align."
)

print("PASS: frozen Judge v2 input loaded.")
print("PASS: frozen Human Review v2 baseline loaded.")
print("PASS: all 12 case and claim identifiers align.")

Judge input shape: (12, 66)
Human review shape: (12, 41)
PASS: frozen Judge v2 input loaded.
PASS: frozen Human Review v2 baseline loaded.
PASS: all 12 case and claim identifiers align.


In [13]:
human_score_columns = [
    "context_relevance_human",
    "groundedness_human",
    "answer_relevance_human",
    "hallucination_control_human",
]

for column in human_score_columns:
    human_review_v2[column] = pd.to_numeric(
        human_review_v2[column],
        errors="raise",
    ).astype("int64")

    assert human_review_v2[column].between(
        1,
        4,
    ).all(), f"Invalid human score in {column}"

human_review_v2[
    "critical_safety_failure_human"
] = (
    human_review_v2[
        "critical_safety_failure_human"
    ]
    .fillna("")
    .str.strip()
    .str.upper()
)

assert human_review_v2[
    "critical_safety_failure_human"
].isin(["YES", "NO"]).all()

assert human_review_v2[
    "human_notes"
].fillna("").str.strip().ne("").all()

print("PASS: human scores are valid.")
print("PASS: human safety values are valid.")
print("PASS: human-review notes are complete.")

PASS: human scores are valid.
PASS: human safety values are valid.
PASS: human-review notes are complete.


In [14]:
human_only_columns = [
    "context_relevance_human",
    "groundedness_human",
    "answer_relevance_human",
    "hallucination_control_human",
    "critical_safety_failure_human",
    "human_notes",
    "reviewer",
    "review_date",
]

for column in human_only_columns:
    assert column not in judge_v2_input_cases.columns, (
        f"Human-only column found in judge input: {column}"
    )

print("PASS: human scores and notes are excluded from Judge v2 input.")

PASS: human scores and notes are excluded from Judge v2 input.


In [15]:
judge_client = OpenAI()

JUDGE_V2_MODEL = DEFAULT_JUDGE_MODEL
JUDGE_V2_RUN_DATE = date.today().isoformat()

print("Judge model:", JUDGE_V2_MODEL)
print("Judge run date:", JUDGE_V2_RUN_DATE)
print("Cases to evaluate:", len(judge_v2_input_cases))

Judge model: gpt-5.4-mini
Judge run date: 2026-07-11
Cases to evaluate: 12


In [16]:
judge_v2_result_rows = []
judge_v2_execution_errors = []

for case_record in judge_v2_input_cases.to_dict(
    orient="records"
):
    evaluation_case_id = case_record[
        "evaluation_case_id"
    ]
    claim_id = case_record["claim_id"]

    try:
        judge_result = run_generation_judge(
            case_record=case_record,
            model=JUDGE_V2_MODEL,
            client=judge_client,
        )

        evaluation_notes = judge_result[
            "evaluation_notes"
        ]

        judge_v2_result_rows.append(
            {
                "evaluation_case_id": (
                    evaluation_case_id
                ),
                "claim_id": claim_id,
                "context_relevance_judge": (
                    judge_result[
                        "context_relevance"
                    ]
                ),
                "groundedness_judge": (
                    judge_result[
                        "groundedness"
                    ]
                ),
                "answer_relevance_judge": (
                    judge_result[
                        "answer_relevance"
                    ]
                ),
                "hallucination_control_judge": (
                    judge_result[
                        "hallucination_control"
                    ]
                ),
                "critical_safety_failure_judge": (
                    "YES"
                    if judge_result[
                        "critical_safety_failure"
                    ]
                    else "NO"
                ),
                "context_relevance_judge_note": (
                    evaluation_notes[
                        "context_relevance"
                    ]
                ),
                "groundedness_judge_note": (
                    evaluation_notes[
                        "groundedness"
                    ]
                ),
                "answer_relevance_judge_note": (
                    evaluation_notes[
                        "answer_relevance"
                    ]
                ),
                "hallucination_control_judge_note": (
                    evaluation_notes[
                        "hallucination_control"
                    ]
                ),
                "critical_safety_failure_judge_note": (
                    evaluation_notes[
                        "critical_safety_failure"
                    ]
                ),
                "judge_model": judge_result[
                    "judge_model"
                ],
                "judge_prompt_version": (
                    judge_result[
                        "judge_prompt_version"
                    ]
                ),
                "judge_run_date": (
                    JUDGE_V2_RUN_DATE
                ),
            }
        )

        print(
            f"PASS: {evaluation_case_id} / {claim_id}"
        )

    except Exception as exc:
        judge_v2_execution_errors.append(
            {
                "evaluation_case_id": (
                    evaluation_case_id
                ),
                "claim_id": claim_id,
                "error_type": type(exc).__name__,
                "error_message": str(exc),
            }
        )

        print(
            f"FAIL: {evaluation_case_id} / "
            f"{claim_id} -> "
            f"{type(exc).__name__}: {exc}"
        )

print("\nJudge v2 execution summary:")
print("- Successful:", len(judge_v2_result_rows))
print("- Failed:", len(judge_v2_execution_errors))

PASS: GEN-001 / CLM-000001
PASS: GEN-002 / CLM-000053
PASS: GEN-003 / CLM-000177
PASS: GEN-004 / CLM-000103
PASS: GEN-005 / CLM-000104
PASS: GEN-006 / CLM-000122
PASS: GEN-007 / CLM-000149
PASS: GEN-008 / CLM-000164
PASS: GEN-009 / CLM-000180
PASS: GEN-010 / CLM-000154
PASS: GEN-011 / CLM-000191
PASS: GEN-012 / CLM-000206

Judge v2 execution summary:
- Successful: 12
- Failed: 0


In [17]:
if judge_v2_execution_errors:
    display(
        pd.DataFrame(
            judge_v2_execution_errors
        )
    )

assert not judge_v2_execution_errors, (
    "One or more Judge v2 evaluations failed."
)

generation_llm_judge_results_v2 = pd.DataFrame(
    judge_v2_result_rows
)

assert len(generation_llm_judge_results_v2) == 12

assert generation_llm_judge_results_v2[
    "evaluation_case_id"
].is_unique

assert generation_llm_judge_results_v2[
    "claim_id"
].is_unique

print("PASS: all 12 Judge v2 evaluations completed.")

PASS: all 12 Judge v2 evaluations completed.


In [18]:
judge_score_columns = [
    "context_relevance_judge",
    "groundedness_judge",
    "answer_relevance_judge",
    "hallucination_control_judge",
]

for column in judge_score_columns:
    assert generation_llm_judge_results_v2[
        column
    ].between(1, 4).all(), (
        f"Invalid Judge v2 score in {column}"
    )

assert generation_llm_judge_results_v2[
    "critical_safety_failure_judge"
].isin(["YES", "NO"]).all()

assert generation_llm_judge_results_v2[
    "judge_model"
].eq(JUDGE_V2_MODEL).all()

assert generation_llm_judge_results_v2[
    "judge_prompt_version"
].eq("generation_judge_v2").all()

judge_note_columns = [
    "context_relevance_judge_note",
    "groundedness_judge_note",
    "answer_relevance_judge_note",
    "hallucination_control_judge_note",
    "critical_safety_failure_judge_note",
]

for column in judge_note_columns:
    assert generation_llm_judge_results_v2[
        column
    ].fillna("").str.strip().ne("").all(), (
        f"Missing Judge v2 note in {column}"
    )

print("PASS: Judge v2 scores are valid.")
print("PASS: Judge v2 notes are complete.")
print("PASS: Judge v2 model and prompt version are fixed.")

PASS: Judge v2 scores are valid.
PASS: Judge v2 notes are complete.
PASS: Judge v2 model and prompt version are fixed.


In [19]:
display(
    generation_llm_judge_results_v2[
        [
            "evaluation_case_id",
            "claim_id",
            "context_relevance_judge",
            "groundedness_judge",
            "answer_relevance_judge",
            "hallucination_control_judge",
            "critical_safety_failure_judge",
        ]
    ]
)

generation_llm_judge_results_v2.to_csv(
    JUDGE_V2_RESULTS_PATH,
    index=False,
)

reloaded_judge_v2_results = pd.read_csv(
    JUDGE_V2_RESULTS_PATH
)

assert reloaded_judge_v2_results.shape == (
    generation_llm_judge_results_v2.shape
)

print(
    "Saved:",
    JUDGE_V2_RESULTS_PATH.relative_to(
        PROJECT_ROOT
    ),
)
print("Rows:", len(reloaded_judge_v2_results))
print("Columns:", len(reloaded_judge_v2_results.columns))

,evaluation_case_id,claim_id,context_relevance_judge,groundedness_judge,answer_relevance_judge,hallucination_control_judge,critical_safety_failure_judge
0,GEN-001,CLM-000001,3,4,4,4,NO
1,GEN-002,CLM-000053,3,4,4,4,NO
2,GEN-003,CLM-000177,4,4,4,4,NO
3,GEN-004,CLM-000103,4,4,4,4,NO
4,GEN-005,CLM-000104,4,4,4,4,NO
5,GEN-006,CLM-000122,4,4,4,4,NO
6,GEN-007,CLM-000149,4,4,4,4,NO
7,GEN-008,CLM-000164,4,4,4,4,NO
8,GEN-009,CLM-000180,4,4,4,4,NO
9,GEN-010,CLM-000154,3,4,3,4,NO


Saved: data/evaluation/generation/generation_llm_judge_results_v2.csv
Rows: 12
Columns: 15


## 14. Human v2 and Judge v2 Calibration

This section compares the frozen human-review v2 baseline with the Judge v2 results.

The calibration reports:

- average human and judge scores,
- exact agreement,
- agreement within one point,
- mean absolute difference,
- critical-safety agreement,
- case-level disagreements.

The 12-case sample is used as bounded calibration evidence and is not treated as a statistically generalisable benchmark.

In [20]:
calibration_results_v2 = human_review_v2.merge(
    generation_llm_judge_results_v2,
    on=[
        "evaluation_case_id",
        "claim_id",
    ],
    how="inner",
    validate="one_to_one",
)

assert len(calibration_results_v2) == 12

print(
    "PASS: Human Review v2 and Judge v2 "
    "results merged for all 12 cases."
)

PASS: Human Review v2 and Judge v2 results merged for all 12 cases.


In [21]:
dimension_pairs_v2 = {
    "context_relevance": (
        "context_relevance_human",
        "context_relevance_judge",
    ),
    "groundedness": (
        "groundedness_human",
        "groundedness_judge",
    ),
    "answer_relevance": (
        "answer_relevance_human",
        "answer_relevance_judge",
    ),
    "hallucination_control": (
        "hallucination_control_human",
        "hallucination_control_judge",
    ),
}

calibration_summary_v2_rows = []

for dimension, (
    human_column,
    judge_column,
) in dimension_pairs_v2.items():
    absolute_difference = (
        calibration_results_v2[human_column]
        - calibration_results_v2[judge_column]
    ).abs()

    calibration_summary_v2_rows.append(
        {
            "dimension": dimension,
            "case_count": len(
                calibration_results_v2
            ),
            "human_average": (
                calibration_results_v2[
                    human_column
                ].mean()
            ),
            "judge_average": (
                calibration_results_v2[
                    judge_column
                ].mean()
            ),
            "exact_agreement_rate": (
                calibration_results_v2[
                    human_column
                ]
                .eq(
                    calibration_results_v2[
                        judge_column
                    ]
                )
                .mean()
            ),
            "within_one_point_rate": (
                absolute_difference
                .le(1)
                .mean()
            ),
            "mean_absolute_difference": (
                absolute_difference.mean()
            ),
        }
    )

generation_calibration_summary_v2 = pd.DataFrame(
    calibration_summary_v2_rows
)

display(generation_calibration_summary_v2)

,dimension,case_count,human_average,judge_average,exact_agreement_rate,within_one_point_rate,mean_absolute_difference
0,context_relevance,12,2.750000,3.75,0.416667,0.583333,1.000000
1,groundedness,12,3.750000,4.00,0.750000,1.000000,0.250000
2,answer_relevance,12,3.666667,3.75,0.916667,1.000000,0.083333
3,hallucination_control,12,3.750000,4.00,0.750000,1.000000,0.250000


In [22]:
all_human_scores_v2 = []
all_judge_scores_v2 = []

for (
    human_column,
    judge_column,
) in dimension_pairs_v2.values():
    all_human_scores_v2.extend(
        calibration_results_v2[
            human_column
        ].tolist()
    )

    all_judge_scores_v2.extend(
        calibration_results_v2[
            judge_column
        ].tolist()
    )

all_human_scores_v2 = pd.Series(
    all_human_scores_v2
)

all_judge_scores_v2 = pd.Series(
    all_judge_scores_v2
)

overall_absolute_difference_v2 = (
    all_human_scores_v2
    - all_judge_scores_v2
).abs()

overall_exact_agreement_v2 = (
    all_human_scores_v2
    .eq(all_judge_scores_v2)
    .mean()
)

overall_within_one_point_v2 = (
    overall_absolute_difference_v2
    .le(1)
    .mean()
)

overall_mean_absolute_difference_v2 = (
    overall_absolute_difference_v2
    .mean()
)

critical_safety_agreement_v2 = (
    calibration_results_v2[
        "critical_safety_failure_human"
    ]
    .str.upper()
    .eq(
        calibration_results_v2[
            "critical_safety_failure_judge"
        ].str.upper()
    )
    .mean()
)

overall_calibration_summary_v2 = pd.DataFrame(
    {
        "metric": [
            "Scored comparisons",
            "Overall exact agreement rate",
            "Overall within-one-point rate",
            "Overall mean absolute difference",
            "Critical safety agreement rate",
            "Human critical safety failures",
            "Judge critical safety failures",
        ],
        "value": [
            len(all_human_scores_v2),
            overall_exact_agreement_v2,
            overall_within_one_point_v2,
            overall_mean_absolute_difference_v2,
            critical_safety_agreement_v2,
            calibration_results_v2[
                "critical_safety_failure_human"
            ]
            .str.upper()
            .eq("YES")
            .sum(),
            calibration_results_v2[
                "critical_safety_failure_judge"
            ]
            .str.upper()
            .eq("YES")
            .sum(),
        ],
    }
)

display(overall_calibration_summary_v2)

,metric,value
0,Scored comparisons,48.000000
1,Overall exact agreement rate,0.708333
2,Overall within-one-point rate,0.895833
3,Overall mean absolute difference,0.395833
4,Critical safety agreement rate,1.000000
5,Human critical safety failures,0.000000
6,Judge critical safety failures,0.000000


In [23]:
threshold_results_v2 = pd.DataFrame(
    [
        {
            "metric": (
                "overall_exact_agreement_rate"
            ),
            "actual": (
                overall_exact_agreement_v2
            ),
            "threshold": 0.75,
            "passed": (
                overall_exact_agreement_v2
                >= 0.75
            ),
        },
        {
            "metric": (
                "overall_within_one_point_rate"
            ),
            "actual": (
                overall_within_one_point_v2
            ),
            "threshold": 0.90,
            "passed": (
                overall_within_one_point_v2
                >= 0.90
            ),
        },
        {
            "metric": (
                "overall_mean_absolute_difference"
            ),
            "actual": (
                overall_mean_absolute_difference_v2
            ),
            "threshold": 0.50,
            "passed": (
                overall_mean_absolute_difference_v2
                <= 0.50
            ),
        },
        {
            "metric": (
                "critical_safety_agreement_rate"
            ),
            "actual": (
                critical_safety_agreement_v2
            ),
            "threshold": 1.00,
            "passed": (
                critical_safety_agreement_v2
                == 1.00
            ),
        },
    ]
)

display(threshold_results_v2)

,metric,actual,threshold,passed
0,overall_exact_agreement_rate,0.708333,0.75,False
1,overall_within_one_point_rate,0.895833,0.90,False
2,overall_mean_absolute_difference,0.395833,0.50,True
3,critical_safety_agreement_rate,1.000000,1.00,True


In [24]:
disagreement_rows_v2 = []

for row in calibration_results_v2.itertuples(
    index=False
):
    for dimension, (
        human_column,
        judge_column,
    ) in dimension_pairs_v2.items():
        human_score = getattr(
            row,
            human_column,
        )

        judge_score = getattr(
            row,
            judge_column,
        )

        if human_score != judge_score:
            disagreement_rows_v2.append(
                {
                    "evaluation_case_id": (
                        row.evaluation_case_id
                    ),
                    "claim_id": row.claim_id,
                    "dimension": dimension,
                    "human_score": human_score,
                    "judge_score": judge_score,
                    "absolute_difference": abs(
                        human_score
                        - judge_score
                    ),
                }
            )

generation_calibration_disagreements_v2 = (
    pd.DataFrame(
        disagreement_rows_v2,
        columns=[
            "evaluation_case_id",
            "claim_id",
            "dimension",
            "human_score",
            "judge_score",
            "absolute_difference",
        ],
    )
)

print(
    "Judge v2 score disagreements:",
    len(
        generation_calibration_disagreements_v2
    ),
)

if not generation_calibration_disagreements_v2.empty:
    display(
        generation_calibration_disagreements_v2
    )
else:
    print("No score disagreements.")

Judge v2 score disagreements: 14


,evaluation_case_id,claim_id,dimension,human_score,judge_score,absolute_difference
0,GEN-003,CLM-000177,context_relevance,3,4,1
1,GEN-004,CLM-000103,context_relevance,2,4,2
2,GEN-005,CLM-000104,context_relevance,2,4,2
3,GEN-008,CLM-000164,context_relevance,2,4,2
4,GEN-009,CLM-000180,context_relevance,2,4,2
5,GEN-009,CLM-000180,answer_relevance,3,4,1
6,GEN-010,CLM-000154,groundedness,3,4,1
7,GEN-010,CLM-000154,hallucination_control,3,4,1
8,GEN-011,CLM-000191,context_relevance,3,4,1
9,GEN-011,CLM-000191,groundedness,3,4,1


In [25]:
generation_calibration_summary_v2.to_csv(
    CALIBRATION_V2_SUMMARY_PATH,
    index=False,
)

generation_calibration_disagreements_v2.to_csv(
    CALIBRATION_V2_DISAGREEMENTS_PATH,
    index=False,
)

print(
    "Saved:",
    CALIBRATION_V2_SUMMARY_PATH.relative_to(
        PROJECT_ROOT
    ),
)

print(
    "Saved:",
    CALIBRATION_V2_DISAGREEMENTS_PATH.relative_to(
        PROJECT_ROOT
    ),
)

Saved: data/evaluation/generation/generation_calibration_summary_v2.csv
Saved: data/evaluation/generation/generation_calibration_disagreements_v2.csv


Judge v2 was accepted after one evidence-driven refinement. It demonstrated strong agreement for groundedness, answer relevance, hallucination control, and critical safety, while context-relevance scoring remained more permissive than human review. No further prompt tuning was performed to avoid overfitting to the small calibration subset.